# 🎰 21 · Bayesian learning and MCMC
### From point estimates to whole posterior distributions: conjugate priors, Bayesian linear regression, Gibbs sampling and Metropolis–Hastings

**BiologyToML** · Part E — Bayesian learning, neural networks and model selection · Module 21 of 24

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/21_bayesian_learning_mcmc.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, pandas, Matplotlib, scikit-learn (for checking only) |
| **Prerequisites** | Modules 02 (probability, Bayes' rule), 05 (likelihood, MLE, MAP), 06 (linear regression), 07 (ridge), 09 (logistic regression), 13 (mixture models and EM) |
| **Estimated time** | 150 min (reading + running + the ✏️ exercises) |
| **02-620 lectures covered** | `013_Bayesian-MCMC_mz_032725` slides 1–47 — see [`SLIDE_MAP.md`](../SLIDE_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Real data: the prostate-cancer data of Stamey et al. (1989) (97 men, 8 clinical predictors of log PSA), used for
> Bayesian linear regression, direct sampling and Gibbs sampling. SIMULATED data (labelled as such): allele counts, a
> genotype–phenotype study, a case–control study and a two-cell-type expression mixture. Slide numbers refer to the full
> 47-slide deck; slides 5, 11–13, 26–32, 34 and 36–38 are hidden in the lecture PDF but are part of the deck.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Distinguish** maximum-likelihood, MAP and fully Bayesian estimation, and **explain** what a posterior distribution adds to a point estimate.
2. **Compute** a conjugate Beta–binomial posterior, a credible interval and a posterior predictive distribution for an allele frequency.
3. **Derive** the Gaussian posterior of Bayesian linear regression, $\Sigma=(X^\top X/\sigma_e^2+I/\sigma_w^2)^{-1}$, $\mu=\Sigma X^\top y/\sigma_e^2$, and **connect** its mean to ridge regression.
4. **Use** posterior samples to compute any quantity of interest — intervals, probabilities, predictions — as an average.
5. **Implement** Gibbs sampling from scratch (bivariate Gaussian, Bayesian linear regression with known and unknown noise variance, a Gaussian mixture) and **verify** it against closed-form answers.
6. **Diagnose** MCMC output: burn-in, trace plots, running averages, autocorrelation, effective sample size and several chains.
7. **Implement** random-walk Metropolis–Hastings for a logistic-regression posterior and **explain** the role of the proposal width.
8. **Connect** Gibbs sampling to coordinate descent and to EM, and to MCMC applications in genetics (phasing, population structure).

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit | 5 min |
| 1 · Three ways to learn a parameter | MLE, MAP and the full posterior for an allele frequency | 10 min |
| 2 · Conjugate priors: Beta–binomial | closed-form posterior, credible interval, predictive | 15 min |
| 3 · Bayesian linear regression | posterior $\mathcal N(\mu,\Sigma)$, ridge = MAP, linked SNPs, predictive bands | 20 min |
| 4 · The general recipe | model → prior → posterior; when closed forms fail | 7 min |
| 5 · Sampling from the posterior | real prostate data: any statistic is an average | 10 min |
| 6 · Gibbs sampling | one coordinate at a time; correlated targets; unknown noise | 25 min |
| 7 · Why it works and how to check | coordinate descent, burn-in, autocorrelation, ESS, $\hat R$ | 18 min |
| 8 · Metropolis–Hastings | logistic regression for a risk allele; proposal width | 18 min |
| 9 · Gibbs for mixtures and MCMC in biology | the sampling sibling of EM | 12 min |
| 10 · Summary, checkpoint, discussion | slide 47; self-test | 10 min |

## 🧫 The biological question

A clinical geneticist sequences ten patients and sees a candidate variant on one of their twenty chromosomes. *How common
is the variant?* The maximum-likelihood answer, 1/20 = 5 %, is a single number that hides how little twenty chromosomes
tell us — and had the variant not been seen at all, the MLE would declare it absent. A plant breeder estimates the effect
of a SNP on grain yield from 30 lines; a urologist asks how much a patient's tumour volume raises his PSA level. In each
case the useful answer is not one number but a statement about **uncertainty**: "the frequency is between about 2 % and
17 % with 95 % probability", "the probability that this allele *increases* risk is 99 %" — both are answers this notebook
will compute (Sections 2 and 8).

**Bayesian learning** delivers exactly such statements: it turns prior knowledge plus data into a whole **posterior
distribution** over the unknown quantities. The price is computation: except in a few textbook cases, the posterior has no
formula. **Markov chain Monte Carlo (MCMC)** pays that price by *sampling* from the posterior instead of writing it
down — and MCMC is the engine behind widely used tools in genetics, such as STRUCTURE for population structure (Pritchard
et al. 2000), PHASE for haplotype reconstruction (Stephens et al. 2001) and MrBayes for phylogenetic trees (Huelsenbeck &
Ronquist 2001).

This notebook completes the course's estimation trilogy — MLE (Module 05), MAP (Modules 05 and 07), and now the fully
Bayesian approach — and builds the samplers from scratch, checking each against an exact answer.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import Ridge

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
They load the prostate-cancer data, simulate genotypes and quantitative traits, and provide the logistic function, column z-scoring and a numerically stable log-sum-exp. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import io
import numpy as np
import pandas as pd

PROSTATE_CSV = """lcavol,lweight,age,lbph,svi,lcp,gleason,pgg45,lpsa,train
-0.579818,2.76946,50,-1.38629,0,-1.38629,6,0,-0.430783,T
-0.994252,3.31963,58,-1.38629,0,-1.38629,6,0,-0.162519,T
-0.510826,2.69124,74,-1.38629,0,-1.38629,7,20,-0.162519,T
-1.20397,3.28279,58,-1.38629,0,-1.38629,6,0,-0.162519,T
0.751416,3.43237,62,-1.38629,0,-1.38629,6,0,0.371564,T
-1.04982,3.22883,50,-1.38629,0,-1.38629,6,0,0.765468,T
0.737164,3.47352,64,0.615186,0,-1.38629,6,0,0.765468,F
0.693147,3.53951,58,1.53687,0,-1.38629,6,0,0.854415,T
-0.776529,3.53951,47,-1.38629,0,-1.38629,6,0,1.04732,F
0.223144,3.24454,63,-1.38629,0,-1.38629,6,0,1.04732,F
0.254642,3.60414,65,-1.38629,0,-1.38629,6,0,1.26695,T
-1.34707,3.59868,63,1.26695,0,-1.38629,6,0,1.26695,T
1.61343,3.02286,63,-1.38629,0,-0.597837,7,30,1.26695,T
1.47705,2.99823,67,-1.38629,0,-1.38629,7,5,1.34807,T
1.20597,3.44202,57,-1.38629,0,-0.430783,7,5,1.39872,F
1.54116,3.06105,66,-1.38629,0,-1.38629,6,0,1.44692,T
-0.415515,3.51601,70,1.24415,0,-0.597837,7,30,1.47018,T
2.28849,3.64936,66,-1.38629,0,0.371564,6,0,1.4929,T
-0.562119,3.26767,41,-1.38629,0,-1.38629,6,0,1.55814,T
0.182322,3.82538,70,1.65823,0,-1.38629,6,0,1.59939,T
1.1474,3.41936,59,-1.38629,0,-1.38629,6,0,1.639,T
2.05924,3.50104,60,1.47476,0,1.34807,7,20,1.65823,F
-0.544727,3.37588,59,-0.798508,0,-1.38629,6,0,1.69562,T
1.78171,3.45157,63,0.438255,0,1.17866,7,60,1.7138,T
0.385262,3.6674,69,1.59939,0,-1.38629,6,0,1.73166,F
1.44692,3.12457,68,0.300105,0,-1.38629,6,0,1.76644,F
0.512824,3.71965,65,-1.38629,0,-0.798508,7,70,1.80006,T
-0.400478,3.86598,67,1.81645,0,-1.38629,7,20,1.81645,F
1.04028,3.12895,67,0.223144,0,0.0487902,7,80,1.84845,T
2.40964,3.37588,65,-1.38629,0,1.61939,6,0,1.89462,T
0.285179,4.09017,65,1.96291,0,-0.798508,6,0,1.92425,T
0.182322,3.80444,65,1.70475,0,-1.38629,6,0,2.00821,F
1.27536,3.03735,71,1.26695,0,-1.38629,6,0,2.00821,T
0.00995033,3.26767,54,-1.38629,0,-1.38629,6,0,2.02155,F
-0.0100503,3.21687,63,-1.38629,0,-0.798508,6,0,2.04769,T
1.30833,4.11985,64,2.17134,0,-1.38629,7,5,2.08567,F
1.42311,3.65713,73,-0.579819,0,1.65823,8,15,2.15756,T
0.457425,2.37491,64,-1.38629,0,-1.38629,7,15,2.19165,T
2.66096,4.08514,68,1.37372,1,1.83258,7,35,2.21375,T
0.797507,3.01308,56,0.936093,0,-0.162519,7,5,2.27727,T
0.620576,3.142,60,-1.38629,0,-1.38629,9,80,2.29757,T
1.4422,3.68261,68,-1.38629,0,-1.38629,7,10,2.30757,F
0.582216,3.86598,62,1.7138,0,-0.430783,6,0,2.32728,T
1.77156,3.89691,61,-1.38629,0,0.81093,7,6,2.37491,F
1.48614,3.4095,66,1.7492,0,-0.430783,7,20,2.52172,T
1.66393,3.39283,61,0.615186,0,-1.38629,7,15,2.55334,T
2.72785,3.99545,79,1.87947,1,2.65676,9,100,2.56879,T
1.16315,4.03512,68,1.7138,0,-0.430783,7,40,2.56879,F
1.74572,3.49802,43,-1.38629,0,-1.38629,6,0,2.59152,F
1.22083,3.56812,70,1.37372,0,-0.798508,6,0,2.59152,F
1.09192,3.9936,68,-1.38629,0,-1.38629,7,50,2.65676,T
1.66013,4.23483,64,2.07317,0,-1.38629,6,0,2.67759,T
0.512824,3.63363,64,1.4929,0,0.0487902,7,70,2.68444,F
2.12704,4.12147,68,1.76644,0,1.44692,7,40,2.69124,F
3.15359,3.51601,59,-1.38629,0,-1.38629,7,5,2.70471,F
1.26695,4.28013,66,2.12226,0,-1.38629,7,15,2.718,T
0.97456,2.86505,47,-1.38629,0,0.500775,7,4,2.78809,F
0.463734,3.76468,49,1.42311,0,-1.38629,6,0,2.79423,T
0.542324,4.17823,70,0.438255,0,-1.38629,7,20,2.80639,T
1.06126,3.85121,61,1.29473,0,-1.38629,7,40,2.81241,T
0.457425,4.5245,73,2.3263,0,-1.38629,6,0,2.842,T
1.99742,3.71965,63,1.61939,1,1.90954,7,40,2.85359,F
2.77571,3.52489,72,-1.38629,0,1.55814,9,95,2.85359,T
2.03471,3.91701,66,2.00821,1,2.11021,7,60,2.882,F
2.07317,3.62301,64,-1.38629,0,-1.38629,6,0,2.882,F
1.45862,3.83622,61,1.32176,0,-0.430783,7,20,2.88759,F
2.02287,3.87847,68,1.78339,0,1.32176,7,70,2.92047,T
2.19834,4.05091,72,2.30757,0,-0.430783,7,10,2.96269,T
-0.446287,4.40855,69,-1.38629,0,-1.38629,6,0,2.96269,T
1.19392,4.78038,72,2.3263,0,-0.798508,7,5,2.97298,T
1.86408,3.59319,60,-1.38629,1,1.32176,7,60,3.01308,T
1.16002,3.34109,77,1.7492,0,-1.38629,7,25,3.03735,T
1.21491,3.82538,69,-1.38629,1,0.223144,7,20,3.05636,F
1.83896,3.23672,60,0.438255,1,1.17866,9,90,3.07501,F
2.99923,3.84908,69,-1.38629,1,1.90954,7,20,3.27526,T
3.14113,3.26385,68,-0.0512933,1,2.42037,7,50,3.33755,T
2.01089,4.43379,72,2.12226,0,0.500775,7,60,3.39283,T
2.53766,4.35478,78,2.3263,0,-1.38629,7,10,3.4356,T
2.6483,3.58213,69,-1.38629,1,2.584,7,70,3.45789,T
2.77944,3.82319,63,-1.38629,0,0.371564,7,50,3.51304,F
1.46787,3.07038,66,0.559616,0,0.223144,7,40,3.51601,T
2.51366,3.47352,57,0.438255,0,2.32728,7,60,3.53076,T
2.61301,3.88875,77,-0.527633,1,0.559616,7,30,3.5653,T
2.67759,3.83838,65,1.11514,0,1.7492,9,70,3.57094,F
1.56235,3.70991,60,1.69562,0,0.81093,7,30,3.58768,T
3.30285,3.51898,64,-1.38629,1,2.32728,7,60,3.63099,T
2.02419,3.7317,58,1.639,0,-1.38629,6,0,3.68009,T
1.73166,3.36902,62,-1.38629,1,0.300105,7,30,3.71235,T
2.80759,4.71805,65,-1.38629,1,2.46385,7,60,3.98434,T
1.56235,3.69511,76,0.936093,1,0.81093,7,75,3.9936,T
3.24649,4.10182,68,-1.38629,0,-1.38629,6,0,4.02981,T
2.5329,3.67757,61,1.34807,1,-1.38629,7,15,4.12955,T
2.83027,3.8764,68,-1.38629,1,1.32176,7,60,4.38515,T
3.821,3.89691,44,-1.38629,1,2.16905,7,40,4.68444,T
2.90745,3.39619,52,-1.38629,1,2.46385,7,10,5.14312,F
2.88256,3.77391,68,1.55814,1,1.55814,7,80,5.47751,T
3.47197,3.975,68,0.438255,1,2.90417,7,20,5.58293,F
"""


def load_prostate() -> pd.DataFrame:
    """Prostate cancer data of 02-620 Lecture 2 (97 men, 8 clinical predictors, outcome ``lpsa``)."""
    df = pd.read_csv(io.StringIO(PROSTATE_CSV))
    df["train"] = df["train"] == "T"
    return df


def simulate_genotypes(rng: np.random.Generator, n_samples: int, n_snps: int, n_pops: int = 1,
                       fst: float = 0.0, maf_range=(0.05, 0.5)):
    """Simulate 0/1/2 genotypes under Hardy–Weinberg equilibrium.

    Each SNP gets an ancestral allele frequency drawn uniformly from ``maf_range``.  With
    ``n_pops > 1`` and ``fst > 0`` every population's frequency is drawn from the
    Balding–Nichols model Beta(p(1−F)/F, (1−p)(1−F)/F).  Samples are split evenly across
    populations.  Returns ``(G, pop, freqs)``: the N × J genotype matrix, the population of
    each sample and the n_pops × J matrix of population allele frequencies.

    ``G`` is stored as int8 to save memory: convert with ``G.astype(float)`` before arithmetic
    whose intermediate results can exceed 127.  A rare SNP can come out monomorphic (no
    variation) in a small sample — skip such columns in per-SNP tests.
    """
    p = rng.uniform(*maf_range, n_snps)
    if n_pops > 1 and fst > 0:
        a, b = p * (1 - fst) / fst, (1 - p) * (1 - fst) / fst
        freqs = np.clip(rng.beta(a, b, size=(n_pops, n_snps)), 0.001, 0.999)
    else:
        freqs = np.tile(p, (n_pops, 1))
    pop = np.repeat(np.arange(n_pops), int(np.ceil(n_samples / n_pops)))[:n_samples]
    G = rng.binomial(2, freqs[pop]).astype(np.int8)
    return G, pop, freqs


def simulate_quantitative_trait(rng: np.random.Generator, G: np.ndarray, n_causal: int = 5,
                                h2: float = 0.5, causal=None):
    """Simulate a quantitative phenotype y = G·β + ε with ``n_causal`` causal SNPs.

    The genetic part is scaled so that it explains a fraction ``h2`` (the heritability) of
    the phenotypic variance.  Returns ``(y, beta, causal_idx)``.
    """
    N, J = G.shape
    causal_idx = np.sort(rng.choice(J, n_causal, replace=False) if causal is None else np.asarray(causal))
    beta = np.zeros(J)
    beta[causal_idx] = rng.choice([-1, 1], len(causal_idx)) * rng.uniform(0.5, 1.0, len(causal_idx))
    g = G @ beta
    scale = np.sqrt(h2 / g.var()) if g.var() > 0 else 0.0
    beta *= scale
    y = G @ beta + rng.normal(0, np.sqrt(1 - h2), N)
    return y, beta, causal_idx


def sigmoid(z):
    """Logistic function 1 / (1 + e^(−z)), written via tanh so that it never overflows."""
    return 0.5 * (1.0 + np.tanh(0.5 * np.asarray(z, float)))


def logsumexp(a, axis=None):
    """log(Σ exp(a)) computed stably (subtract the maximum first)."""
    a = np.asarray(a, float)
    m = np.max(a, axis=axis, keepdims=True)
    m = np.where(np.isfinite(m), m, 0.0)
    out = np.log(np.sum(np.exp(a - m), axis=axis, keepdims=True)) + m
    return np.squeeze(out, axis=axis) if axis is not None else out.item()


def standardize(X, mean=None, sd=None):
    """Z-score the columns of X. Pass the training mean/sd to transform test data the same way.

    Returns ``(Z, mean, sd)``; constant columns get sd = 1 so they map to 0.
    """
    X = np.asarray(X, float)
    mean = X.mean(axis=0) if mean is None else mean
    sd = X.std(axis=0) if sd is None else sd
    sd = np.where(sd > 0, sd, 1.0)
    return (X - mean) / sd, mean, sd


print("🧰 Toolkit loaded:", ", ".join(['load_prostate', 'simulate_genotypes', 'simulate_quantitative_trait', 'sigmoid', 'logsumexp', 'standardize']))

---
## 1 · Three ways to learn a parameter (slides 2–6)

> 📍 **Where we are.** Every probabilistic model in this course has followed the same two-step template: specify a
> model, then estimate its parameters. Module 05 introduced two estimators; this lecture adds the third (slide 4).

| Strategy (slides 2–4) | What $\theta$ is | What we compute |
|---|---|---|
| **Maximum likelihood (MLE)** — frequentist | a fixed, unknown number | $\hat\theta_{\text{MLE}}=\arg\max_\theta\log P(\text{data}\mid\theta)$ |
| **Maximum a posteriori (MAP)** — Bayesian | a random variable with a prior | $\hat\theta_{\text{MAP}}=\arg\max_\theta\log P(\theta\mid\text{data})$ |
| **Fully Bayesian** — new! | a random variable with a prior | the whole distribution $P(\theta\mid\text{data})$ |

The bridge between prior and posterior is **Bayes' rule** (slide 6):

$$P(\theta\mid D)=\frac{P(D\mid\theta)\,P(\theta)}{P(D)},\qquad P(D)=\int P(D\mid\theta)\,P(\theta)\,d\theta.$$

*In words:* the **posterior** belief about $\theta$ after seeing data $D$ is the **likelihood** (how probable the data are
if $\theta$ were true) times the **prior** belief before the data, divided by the **evidence** $P(D)$, which only rescales
the result so that it integrates to 1.

🧮 **Math refresher — integrals as areas.** For a continuous parameter, probabilities are areas under a density curve,
and $\int f(\theta)\,d\theta$ means "the area under $f$". On a computer we approximate it by slicing the axis into many
narrow strips of width $\Delta\theta$ and adding up $f(\theta)\Delta\theta$ — a **grid approximation**. That is how we
compute $P(D)$ below.

Our first example: a variant seen on $k=1$ of $n=20$ chromosomes. The likelihood is binomial,
$P(k\mid\theta)=\binom nk\theta^k(1-\theta)^{n-k}$ (Module 02). For the prior we suppose that similar variants in a
reference database typically have frequencies around 10 %, which we encode as a **Beta(3, 27)** distribution (mean
$3/30=0.1$; the Beta distribution is explained in Section 2).

In [ ]:
# ── 1.1 Three answers to one question: how common is this variant? ──
k, n = 1, 20                                # the variant was seen on 1 of 20 sequenced chromosomes
a_pri, b_pri = 3, 27                        # prior Beta(3, 27): similar variants have frequencies around 10 %
theta = np.linspace(0, 0.5, 5001)           # a fine grid of possible allele frequencies
dtheta = theta[1] - theta[0]
likelihood = stats.binom.pmf(k, n, theta)
prior = stats.beta.pdf(theta, a_pri, b_pri)
evidence = np.sum(likelihood * prior) * dtheta             # P(D) = ∫ P(D|θ) P(θ) dθ, by the grid
posterior = likelihood * prior / evidence                  # Bayes' rule on the grid
mle, map_est = theta[np.argmax(likelihood)], theta[np.argmax(posterior)]
post_mean = np.sum(theta * posterior) * dtheta
print(f"MLE = {mle:.4f}   MAP = {map_est:.4f}   posterior mean = {post_mean:.4f}")
print(f"area under the posterior = {np.sum(posterior) * dtheta:.4f}")
print(f"P(θ > 0.10 | data) = {np.sum(posterior[theta > 0.10]) * dtheta:.3f}")

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(theta, likelihood / (np.sum(likelihood) * dtheta), color="tab:blue", label="likelihood P(D|θ) (rescaled to area 1)")
ax.plot(theta, prior, color="tab:green", ls="--", label="prior P(θ) = Beta(3, 27)")
ax.plot(theta, posterior, color="tab:red", lw=2, label="posterior P(θ|D)")
top = ax.get_ylim()[1]
for x, lab, c in [(mle, "MLE", "tab:blue"), (map_est, "MAP", "tab:red"), (post_mean, "posterior mean", "k")]:
    ax.axvline(x, color=c, ls=":", lw=1)
    ax.text(x + 0.003, 0.97 * top, lab, rotation=90, va="top", fontsize=8, color=c)
ax.set_xlabel("allele frequency θ"); ax.set_ylabel("density"); ax.set_xlim(0, 0.4)
ax.set_title("SIMULATED example: variant seen on 1 of 20 chromosomes"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The three estimators give three different numbers: the MLE (0.05) listens only to the data,
the MAP (0.0625) and the posterior mean (0.08) are pulled towards the prior's 10 %. The posterior curve (red) sits between
the likelihood (blue) and the prior (green) and is narrower than both: combining two sources of information reduces
uncertainty. Only the full posterior answers questions like "how probable is it that the variant is more common than
10 %?" — here about 0.26 (printed above). That is the *fully Bayesian* answer of slide 4.

---
## 2 · Conjugate priors: the Beta–binomial model

> 📍 **Where we are.** In Section 1 we needed a grid to normalise the posterior. For some prior–likelihood pairs the
> posterior has a known formula — the "easy case with closed-form solutions" of slide 17, which lists two examples:
> Gaussian likelihood + Gaussian prior, and **binomial likelihood + Beta prior**.

The **Beta distribution** on $[0,1]$ has density

$$\text{Beta}(\theta;a,b)=\frac{\theta^{a-1}(1-\theta)^{b-1}}{B(a,b)},\qquad \text{mean}=\frac{a}{a+b},\qquad\text{mode}=\frac{a-1}{a+b-2}.$$

*In words:* a flexible bump on the interval from 0 to 1; you can read $a-1$ and $b-1$ as "pseudo-counts" of carriers and
non-carriers seen before the data ($B(a,b)$ is just the constant that makes the area 1).

Multiply the binomial likelihood by the Beta prior and keep only the parts that depend on $\theta$:

$$P(\theta\mid k)\;\propto\;\theta^{k}(1-\theta)^{n-k}\cdot\theta^{a-1}(1-\theta)^{b-1}=\theta^{a+k-1}(1-\theta)^{b+n-k-1}
\;\;\Rightarrow\;\;\theta\mid k\sim\text{Beta}(a+k,\;b+n-k).$$

*In words:* the posterior is again a Beta distribution — the prior with the observed carriers added to $a$ and the
non-carriers added to $b$. A prior with this property is called **conjugate** to the likelihood. The symbol $\propto$
("proportional to") means "equal up to a constant factor that does not depend on $\theta$".

In [ ]:
# ── 2.1 Conjugacy check, credible intervals, and a comparison with the textbook confidence interval ──
a_post, b_post = a_pri + k, b_pri + n - k
post = stats.beta(a_post, b_post)
print(f"max |grid posterior − Beta({a_post}, {b_post}) density| = {np.abs(posterior - post.pdf(theta)).max():.1e}")
print(f"posterior mean {post.mean():.4f}, mode {(a_post - 1) / (a_post + b_post - 2):.4f}, "
      f"95 % credible interval [{post.ppf(0.025):.4f}, {post.ppf(0.975):.4f}]")

flat = stats.beta(1 + k, 1 + n - k)                        # uniform prior Beta(1, 1)
p_hat = k / n
se = np.sqrt(p_hat * (1 - p_hat) / n)
print(f"\nuniform prior: posterior mode {k / n:.4f} (= MLE), 95 % credible interval "
      f"[{flat.ppf(0.025):.4f}, {flat.ppf(0.975):.4f}]")
print(f"textbook (Wald) 95 % confidence interval p̂ ± 1.96·SE = [{p_hat - 1.96 * se:.4f}, {p_hat + 1.96 * se:.4f}]")

🔍 **What just happened?** The grid posterior of Section 1 and the Beta(4, 46) formula agree to many decimal places —
conjugacy works. A **95 % credible interval** is the range that contains the parameter with 95 % posterior probability;
we take the 2.5 % and 97.5 % quantiles. With a *uniform* prior (no prior knowledge), the posterior mode equals the MLE,
but the interval is still sensible, whereas the common textbook "$\hat p\pm1.96\,\text{SE}$" interval reaches below zero
— an impossible allele frequency — because its normal approximation fails for rare events and small samples.

⚠️ A credible interval and a confidence interval answer different questions. "95 % probability that $\theta$ lies in
this interval, given the data and the prior" is a Bayesian statement; a frequentist 95 % confidence interval is a
procedure that covers the true value in 95 % of repeated experiments.

### 2.2 More data, less prior

The posterior mean of a Beta–binomial model can be written as a weighted average:

$$\mathbb E[\theta\mid k]=\frac{a+k}{a+b+n}=\underbrace{\frac{a+b}{a+b+n}}_{\text{weight of prior}}\cdot\frac{a}{a+b}
+\underbrace{\frac{n}{a+b+n}}_{\text{weight of data}}\cdot\frac{k}{n}.$$

*In words:* the prior acts like $a+b$ extra observations; when the real sample size $n$ is much larger, the data dominate
and the posterior mean approaches the MLE.

In [ ]:
# ── 2.2 The posterior as the sample grows (always 5 % carriers) ──
fig, ax = plt.subplots(figsize=(7.5, 3.5))
ax.plot(theta, prior, color="tab:green", ls="--", label="prior Beta(3, 27)")
for n_c, col in zip([20, 200, 2000], ["tab:orange", "tab:red", "tab:purple"]):
    k_c = n_c // 20
    post_c = stats.beta(a_pri + k_c, b_pri + n_c - k_c)
    w_prior = (a_pri + b_pri) / (a_pri + b_pri + n_c)
    check = w_prior * a_pri / (a_pri + b_pri) + (1 - w_prior) * k_c / n_c
    print(f"n = {n_c:5d}, k = {k_c:3d}: posterior mean {post_c.mean():.4f} (weighted-average formula {check:.4f}), "
          f"prior weight {w_prior:.3f}, 95 % CrI [{post_c.ppf(0.025):.4f}, {post_c.ppf(0.975):.4f}]")
    ax.plot(theta, post_c.pdf(theta), color=col, label=f"posterior, n = {n_c}")
ax.axvline(0.05, color="0.4", ls=":", lw=1, label="observed proportion 0.05")
ax.set_xlim(0, 0.25); ax.set_xlabel("allele frequency θ"); ax.set_ylabel("density")
ax.set_title("SIMULATED counts: the prior matters less as data accumulate"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** With 20 chromosomes, the prior still carries 60 % of the weight; with 2,000 it carries
1.5 %, and the posterior is a narrow peak just above 0.05. The weighted-average formula reproduces every posterior mean.
This is the reassuring side of Bayesian learning: with enough data, reasonable people with different priors end up
agreeing.

### 2.3 The posterior predictive distribution

Often the question is about *future data*: what genotype will the next person have? The **posterior predictive**
averages the model's prediction over all plausible parameter values:

$$P(\tilde y\mid D)=\int P(\tilde y\mid\theta)\,P(\theta\mid D)\,d\theta.$$

*In words:* instead of plugging in one "best" $\theta$, ask every plausible $\theta$ for its prediction and average the
answers, weighting each by its posterior probability. For two chromosomes the result is a *beta-binomial* distribution,
and we can also approximate the integral by sampling: draw $\theta$ from the posterior, then a genotype given $\theta$.

In [ ]:
# ── 2.3 Posterior predictive genotype of the next person (two chromosomes) ──
rng = np.random.default_rng(SEED + 2)
g_values = np.array([0, 1, 2])
pred_bayes = stats.betabinom.pmf(g_values, 2, a_post, b_post)    # exact: ∫ Binomial(2, θ) Beta(θ; a', b') dθ
pred_plugin = stats.binom.pmf(g_values, 2, k / n)                 # plug in the MLE θ = 0.05
theta_draws = rng.beta(a_post, b_post, 100_000)                   # sample θ from the posterior …
geno = rng.binomial(2, theta_draws)                               # … then a genotype given θ
pred_mc = np.bincount(geno, minlength=3) / len(geno)
print(pd.DataFrame({"plug-in MLE": pred_plugin, "Bayes (exact)": pred_bayes, "Bayes (100,000 samples)": pred_mc},
                   index=["genotype 0 (no copy)", "genotype 1 (carrier)", "genotype 2 (homozygous)"]).round(4))
print(f"\nP(next chromosome carries the variant | data) = posterior mean = {a_post / (a_post + b_post):.3f}")

🔍 **What just happened?** The sampling approximation matches the exact beta-binomial to within about 0.002 — a first
taste of "any quantity is an average over posterior samples" (Section 5). Compared with plugging in the MLE, the Bayesian
predictive gives more weight to carriers and homozygotes, because it accounts both for the prior and for the uncertainty
about $\theta$.

✏️ **Your turn.** Set `k = 0` (the variant was *not* seen) and re-run 1.1–2.3. What is the MLE? What do the posterior
mean, the 95 % credible interval and $P(\theta>0.01\mid D)$ say instead? Which answer would you report to a clinician?

### 📑 Slide connection — `013_Bayesian-MCMC_mz_032725`, slides 2–6

- **What the slides show.** Three general strategies for learning a probability model: MLE (frequentist, $\theta$ deterministic,
$\hat\theta_{MLE}=\arg\max_\theta\log P(data\mid\theta)$), MAP (Bayesian, $\theta$ random,
$\hat\theta_{MAP}=\arg\max_\theta\log P(\theta\mid data)$) and the fully Bayesian approach, which estimates the full
distribution $P(\theta\mid data)$ by a closed-form solution, direct sampling, Gibbs sampling, etc. (slides 2–4; slide 5,
hidden, repeats this as "so far … new!"); Bayes' rule with prior, likelihood and posterior labelled (slide 6).
- **What this notebook adds.** A worked allele-frequency example in which MLE, MAP and posterior mean differ; grid normalisation of Bayes'
rule; the Beta–binomial conjugate pair (slide 17 names it) with credible intervals, the prior-weight formula and the
posterior predictive, checked by sampling.
- **Exam tip.** Know the Beta–binomial update ($a\to a+k$, $b\to b+n-k$) and the formulas for the posterior mean
$(a+k)/(a+b+n)$ and mode $(a+k-1)/(a+b+n-2)$; with a uniform prior the mode equals the MLE.

---
## 3 · Bayesian linear regression (slides 7–15)

> 📍 **Where we are.** Module 06 fitted $y=Xw+e$ by least squares (the MLE) and Module 07 added a ridge penalty (a MAP
> estimate). Slides 7–15 now treat the regression weights $w$ fully Bayesianly — and, because both the likelihood and the
> prior are Gaussian, the posterior has a closed form.

**Model (slide 7).** $y=Xw+e$ with inputs $X\in\mathbb R^{n\times m}$ and outputs $y\in\mathbb R^n$ (the slides write $n$
samples and $m$ features; in this course $N$ and $J$). Prior on the weights: $w\sim\mathcal N(0,\sigma_w^2I)$ — every
weight independently Gaussian around 0. Noise: $e\sim\mathcal N(0,\sigma_e^2I)$. The variances $\sigma_e^2$ and
$\sigma_w^2$ are treated as known for now.

**Posterior (slide 8).** $P(w\mid y,X)\propto P(y\mid X,w)\,P(w)$ with $P(y\mid X,w)=\mathcal N(Xw,\sigma_e^2I)$ and
$P(w)=\mathcal N(0,\sigma_w^2I)$. Taking logs:

$$\log P(w\mid y,X)=-\frac{1}{2\sigma_e^2}\|y-Xw\|^2-\frac{1}{2\sigma_w^2}\|w\|^2+\text{const}.$$

*In words:* the log-posterior is a least-squares fit term plus a penalty on large weights — exactly the ridge objective
of Module 07 (divided by $-2\sigma_e^2$).

**MAP (slide 9).** Maximising gives

$$\hat w_{\text{MAP}}=\Big(X^\top X+\frac{\sigma_e^2}{\sigma_w^2}I\Big)^{-1}X^\top y .$$

*In words:* ridge regression with penalty $\lambda=\sigma_e^2/\sigma_w^2$. "Small $\sigma_0^2$ value means strong prior
belief" (slide 9; the slide writes $\sigma_0^2$ for $\sigma_w^2$): a narrow prior means a large $\lambda$ and strong
shrinkage towards 0.

**Full posterior (slide 15).** Expand the squares in the log-posterior and collect the terms in $w$:

$$\log P(w\mid y,X)=-\tfrac12\,w^\top\Big(\frac{X^\top X}{\sigma_e^2}+\frac{I}{\sigma_w^2}\Big)w+w^\top\frac{X^\top y}{\sigma_e^2}+\text{const}.$$

*In words:* a quadratic function of $w$ with a negative "curvature matrix" — the log-density of a Gaussian.

🧮 **Math refresher — reading a Gaussian off a quadratic ("completing the square").** A Gaussian $\mathcal N(\mu,\Sigma)$
has log-density $-\tfrac12(w-\mu)^\top\Sigma^{-1}(w-\mu)+\text{const}=-\tfrac12w^\top\Sigma^{-1}w+w^\top\Sigma^{-1}\mu+\text{const}$.
Match the two expressions term by term: the matrix in the quadratic term is $\Sigma^{-1}$ and the vector in the linear term
is $\Sigma^{-1}\mu$. (Module 18 used the same trick for Gaussian conditionals.) Therefore

$$P(w\mid y,X)=\mathcal N(\mu,\Sigma),\qquad \Sigma=\Big(\frac{X^\top X}{\sigma_e^2}+\frac{I}{\sigma_w^2}\Big)^{-1},\qquad
\mu=\Sigma\,\frac{X^\top y}{\sigma_e^2}.$$

*In words:* the posterior precision ($\Sigma^{-1}$) is the data precision $X^\top X/\sigma_e^2$ plus the prior precision
$I/\sigma_w^2$ — information adds up. The posterior mean is the MAP/ridge estimate (multiply $\mu$'s formula through by
$\sigma_e^2$ to see it), and the posterior covariance tells us how uncertain each weight is and how the weights' errors are
correlated.

In [ ]:
# ── 3.1 A SIMULATED genotype–phenotype study: one SNP, 30 individuals ──
rng = np.random.default_rng(SEED + 3)
G1, _, freqs1 = simulate_genotypes(rng, 30, 1, maf_range=(0.3, 0.4))
y1, beta1, _ = simulate_quantitative_trait(rng, G1, n_causal=1, h2=0.3)      # noise variance 1 − h² = 0.7
SIGMA_E2 = 0.7                                      # treated as known, as on slide 7
x_snp = G1[:, 0] - G1[:, 0].mean()                  # centred genotype: no intercept needed (slide 7's model)
y_c = y1 - y1.mean()                                # centred phenotype
print(f"allele frequency {freqs1[0, 0]:.2f}; genotype counts (0/1/2) {np.bincount(G1[:, 0], minlength=3)}; "
      f"true effect per allele w = {beta1[0]:.3f}")


def blr_posterior(X, y, sigma_e2, sigma_w2):
    """Posterior N(μ, Σ) of w for y = Xw + e, e ~ N(0, σe² I), prior w ~ N(0, σw² I) (slide 15)."""
    A = X.T @ X / sigma_e2 + np.eye(X.shape[1]) / sigma_w2      # posterior precision Σ⁻¹
    Sigma = np.linalg.inv(A)
    mu = Sigma @ X.T @ y / sigma_e2
    return mu, Sigma


mu1, S1 = blr_posterior(x_snp[:, None], y_c, SIGMA_E2, sigma_w2=1.0)
ridge = Ridge(alpha=SIGMA_E2 / 1.0, fit_intercept=False).fit(x_snp[:, None], y_c).coef_
ols = np.sum(x_snp * y_c) / np.sum(x_snp ** 2)
print(f"posterior: mean {mu1[0]:.4f}, sd {np.sqrt(S1[0, 0]):.4f}, 95 % credible interval "
      f"[{mu1[0] - 1.96 * np.sqrt(S1[0, 0]):.3f}, {mu1[0] + 1.96 * np.sqrt(S1[0, 0]):.3f}]")
print(f"ridge (λ = σe²/σw²) {ridge[0]:.4f}   OLS {ols:.4f}")

In [ ]:
# ── 3.2 Check the closed form on a grid, and see what the prior width does ──
w_grid = np.linspace(-2.0, 2.0, 4001)
dw = w_grid[1] - w_grid[0]
log_unnorm = (-np.sum((y_c[None, :] - w_grid[:, None] * x_snp[None, :]) ** 2, axis=1) / (2 * SIGMA_E2)
              - w_grid ** 2 / (2 * 1.0))                       # log likelihood + log prior, up to a constant
grid_post = np.exp(log_unnorm - log_unnorm.max())
grid_post /= grid_post.sum() * dw
exact = stats.norm(mu1[0], np.sqrt(S1[0, 0])).pdf(w_grid)
print(f"max |grid posterior − N(μ, Σ) density| = {np.abs(grid_post - exact).max():.1e}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
lik = np.exp(log_unnorm + w_grid ** 2 / 2); lik /= lik.sum() * dw
axes[0].plot(w_grid, stats.norm(0, 1).pdf(w_grid), "--", color="tab:green", label="prior N(0, σw² = 1)")
axes[0].plot(w_grid, lik, color="tab:blue", label="likelihood (rescaled)")
axes[0].plot(w_grid, exact, color="tab:red", lw=2, label="posterior N(μ, Σ)")
axes[0].axvline(beta1[0], color="k", ls=":", lw=1, label="true effect")
axes[0].set_xlabel("effect per allele w (phenotype units)"); axes[0].set_ylabel("density")
axes[0].set_title("SIMULATED: 30 individuals, one SNP"); axes[0].legend(fontsize=7)
for sw2, col in zip([0.01, 0.1, 1.0, 100.0], ["tab:purple", "tab:orange", "tab:red", "tab:gray"]):
    m, S = blr_posterior(x_snp[:, None], y_c, SIGMA_E2, sw2)
    axes[1].plot(w_grid, stats.norm(m[0], np.sqrt(S[0, 0])).pdf(w_grid), color=col,
                 label=f"σw² = {sw2:g}: mean {m[0]:.2f}, sd {np.sqrt(S[0, 0]):.2f}")
axes[1].set_xlabel("effect per allele w"); axes[1].set_ylabel("posterior density")
axes[1].set_title("small σw² = strong prior = strong shrinkage (slide 9)"); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The posterior computed by brute force on a grid coincides with the closed-form Gaussian of
slide 15, and its mean equals the ridge estimate with $\lambda=\sigma_e^2/\sigma_w^2$ (cell 3.1). In the left panel the
posterior is almost the rescaled likelihood: 30 individuals carry much more information than the broad prior
$\mathcal N(0,1)$. The true effect of this simulation (dotted line) lies in the tail of the posterior but inside its
95 % credible interval (cell 3.1): with 30 individuals the estimate is uncertain, and the posterior says so honestly. In
the right panel, shrinking $\sigma_w^2$ pulls the posterior towards 0 and makes it narrower; with
$\sigma_w^2=100$ the prior is essentially flat and the posterior mean is the OLS estimate.

### 3.3 Two linked SNPs: what a posterior *covariance* tells you

Neighbouring SNPs are often in **linkage disequilibrium (LD)** — inherited together, so their genotypes are strongly
correlated. Suppose only SNP A affects the phenotype, but SNP B is a near-copy of A. A point estimate gives two numbers;
the posterior shows *why* they are uncertain.

In [ ]:
# ── 3.3 Two SNPs in strong LD (SIMULATED): the joint posterior and 100 direct samples (slides 19–20) ──
def sample_mvn(mu, Sigma, n_draws: int, rng) -> np.ndarray:
    """n_draws samples (rows) from N(mu, Sigma) via the Cholesky factor: mu + z Lᵀ with z standard normal."""
    L = np.linalg.cholesky(Sigma)
    return mu + rng.standard_normal((n_draws, len(mu))) @ L.T


N2 = 60
G2, _, _ = simulate_genotypes(rng, N2, 1, maf_range=(0.3, 0.4))
g_a = G2[:, 0].astype(float)
g_b = np.where(rng.random(N2) < 0.85, g_a, rng.binomial(2, 0.35, N2))     # SNP B copies A in 85 % of people
X2 = np.c_[g_a - g_a.mean(), g_b - g_b.mean()]
W2_TRUE = np.array([0.5, 0.0])                                            # only SNP A is causal
y2 = X2 @ W2_TRUE + rng.normal(0, 1, N2)
y2 -= y2.mean()
mu2, S2 = blr_posterior(X2, y2, sigma_e2=1.0, sigma_w2=1.0)
draws2 = sample_mvn(mu2, S2, 100, rng)
print(f"genotype correlation (LD r) between SNP A and B: {np.corrcoef(g_a, g_b)[0, 1]:.2f}")
print(f"posterior means {np.round(mu2, 3)}, sds {np.round(np.sqrt(np.diag(S2)), 3)}, "
      f"posterior correlation {S2[0, 1] / np.sqrt(S2[0, 0] * S2[1, 1]):+.2f}")
print(f"from 100 samples: P(w_A > w_B) ≈ {np.mean(draws2[:, 0] > draws2[:, 1]):.2f}, "
      f"P(w_A + w_B > 0) ≈ {np.mean(draws2.sum(axis=1) > 0):.2f}")

ga, gb = np.meshgrid(np.linspace(-0.6, 1.4, 150), np.linspace(-1.0, 1.0, 150))
diff = np.stack([ga - mu2[0], gb - mu2[1]], axis=-1)
dens_ld = np.exp(-0.5 * np.einsum("...i,ij,...j->...", diff, np.linalg.inv(S2), diff))   # unnormalised density
fig, ax = plt.subplots(figsize=(5, 4.2))
ax.contour(ga, gb, dens_ld, levels=[0.05, 0.25, 0.6, 0.9], colors="tab:blue", linewidths=1)
ax.scatter(draws2[:, 0], draws2[:, 1], s=8, color="tab:orange", label="100 posterior samples")
ax.plot(*W2_TRUE, "k*", ms=12, label="true weights")
ax.plot([-0.6, 1.4], [-0.6, 1.4], ":", color="0.5", lw=1, label="w_A = w_B")
ax.set_xlabel("w_A (effect of SNP A)"); ax.set_ylabel("w_B (effect of SNP B)")
ax.set_title("SIMULATED linked SNPs: the posterior is a tilted ellipse"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The posterior is a long ellipse tilted *against* the diagonal: the posterior correlation of
the two weights is strongly negative. The data pin down the **sum** of the two effects well (almost every sample has
$w_A+w_B>0$) but cannot tell which SNP carries it — raising $w_A$ and lowering $w_B$ fits the data almost equally well.
This is the statistical heart of **fine-mapping** in genetic association studies: a point estimate (MAP) hides the
ambiguity, the posterior reveals it, and probabilities such as $P(w_A>w_B\mid D)$ quantify it.

### 3.4 Uncertainty about the line versus scatter of individuals

The posterior over $w$ also gives a **posterior predictive** distribution for a new individual with genotype $x_*$:
$y_*\mid D\sim\mathcal N\big(x_*^\top\mu,\;x_*^\top\Sigma x_*+\sigma_e^2\big)$.

*In words:* the predicted phenotype is uncertain for two reasons — we do not know the exact effect ($x_*^\top\Sigma x_*$,
which shrinks as data accumulate) and individuals scatter around the average even if we knew it perfectly ($\sigma_e^2$,
which never shrinks).

In [ ]:
# ── 3.4 Posterior samples of the regression line and the predictive band (one-SNP data of 3.1) ──
g_mean = G1[:, 0].mean()
gg = np.linspace(-0.2, 2.2, 60)                       # genotype axis (0, 1, 2 copies)
xx = (gg - g_mean)[:, None]                           # centred, as in the model
line_draws = sample_mvn(mu1, S1, 30, rng)             # 30 plausible effects w
mean_pred = xx @ mu1
sd_line = np.sqrt(np.sum((xx @ S1) * xx, axis=1))     # uncertainty about the average line
sd_pred = np.sqrt(sd_line ** 2 + SIGMA_E2)            # plus individual scatter
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.fill_between(gg, mean_pred - 1.96 * sd_pred, mean_pred + 1.96 * sd_pred, color="tab:blue", alpha=0.12,
                label="95 % predictive band (new individual)")
ax.fill_between(gg, mean_pred - 1.96 * sd_line, mean_pred + 1.96 * sd_line, color="tab:blue", alpha=0.3,
                label="95 % band for the average")
for wd in line_draws:
    ax.plot(gg, xx[:, 0] * wd[0], color="tab:red", lw=0.6, alpha=0.4)
ax.scatter(G1[:, 0] + rng.uniform(-0.08, 0.08, len(G1)), y_c, s=14, color="k", zorder=3, label="30 individuals")
ax.set_xticks([0, 1, 2]); ax.set_xlabel("genotype (copies of the allele)"); ax.set_ylabel("phenotype (centred)")
ax.set_title("SIMULATED: red = 30 posterior draws of the regression line"); ax.legend(fontsize=7, loc="lower left")
plt.tight_layout(); plt.show()
print(f"at genotype 2: sd of the average = {sd_line[-1]:.3f}, predictive sd = {sd_pred[-1]:.3f} (σe = {np.sqrt(SIGMA_E2):.3f})")

🔍 **What just happened?** The red lines are 30 equally plausible regression lines drawn from the posterior; together they
form the dark band of uncertainty about the *average* phenotype per genotype, which is narrowest near the mean genotype
(all lines pass through the point (mean genotype, 0) because genotype and phenotype were centred).
The light band adds the individual scatter and is dominated by $\sigma_e$: even a perfectly known SNP effect predicts an
individual's phenotype poorly when heritability is low — a sobering, general lesson for genetic prediction.

✏️ **Your turn.** Change the number of individuals in 3.1 from 30 to 300. Which band shrinks, which one does not, and why?

### 📑 Slide connection — `013_Bayesian-MCMC_mz_032725`, slides 7–15

- **What the slides show.** The model $y=Xw+e$ with $X\in\mathbb R^{n\times m}$, $w\sim\mathcal N(0,\sigma_w^2)$ and
$e\sim\mathcal N(0,\sigma_e^2)$ IID (slide 7); posterior ∝ likelihood × prior (slide 8); the MAP estimate
$\hat w_{MAP}=(X^\top X+\frac{\sigma_e^2}{\sigma_w^2}I)^{-1}X^\top y$ with "small $\sigma_0^2$ value means strong prior belief"
(slide 9); full distribution versus point estimate (slides 10, 14; hidden slides 11–13 recap the MAP estimate and the
same point-versus-distribution contrast); by conjugacy the
posterior is $\mathcal N(\mu,\Sigma)$ with $\mu=(X^\top X/\sigma_e^2+I/\sigma_w^2)^{-1}(X^\top y/\sigma_e^2)$ and
$\Sigma=(X^\top X/\sigma_e^2+I/\sigma_w^2)^{-1}$ (slide 15).
- **What this notebook adds.** The completing-the-square derivation of slide 15, numerical checks (grid posterior; MAP = scikit-learn ridge),
the effect of $\sigma_w^2$, a linked-SNP example showing what the posterior covariance means (fine-mapping), and the
posterior predictive with its two sources of uncertainty. Notation: slide 9 writes $\sigma_0^2$ for the prior variance
called $\sigma_w^2$ elsewhere.
- **Exam tip.** Be able to derive $\Sigma$ and $\mu$ by matching the quadratic and linear terms of the log-posterior, and to
show that $\mu$ equals the ridge solution with $\lambda=\sigma_e^2/\sigma_w^2$.

---
## 4 · The general recipe — and where it gets hard (slides 16–18)

> 📍 **Where we are.** We have two closed-form posteriors (Beta–binomial, Gaussian–Gaussian). Slide 16 states the
> general recipe, and slides 17–18 split the world into easy and hard cases.

**General Bayesian inference (slide 16):**
1. Set up the probability model — any model from this course: mixture models, HMMs, linear regression, Bayesian networks,
   logistic regression, …
2. Choose the prior.
3. Determine the posterior from steps 1 and 2.

**Easy case (slide 17):** a conjugate pair gives a named posterior distribution.

| Likelihood | Conjugate prior | Posterior | Where in this course |
|---|---|---|---|
| binomial (counts of carriers) | Beta | Beta | Section 2; naïve Bayes pseudo-counts (Module 08) |
| Gaussian with known variance (regression) | Gaussian | Gaussian | Section 3 |
| Gaussian variance | inverse-gamma | inverse-gamma | Section 6.3 |
| categorical / multinomial | Dirichlet | Dirichlet | Bayesian-network CPT pseudo-counts (Module 17) |

**Difficult case (slide 18):** for logistic regression, mixtures, HMMs or most realistic models, the posterior is not a
named distribution. A grid approximation (Section 1) works in one or two dimensions, but its cost explodes with the
number of parameters. Slide 18's answer: *describe the posterior with a histogram of samples*.

In [ ]:
# ── 4.1 Why grids do not scale: 100 grid points per parameter ──
for n_params in (1, 2, 3, 8, 20):
    points = 100.0 ** n_params
    print(f"{n_params:2d} parameters: {points:.0e} grid points → at 10⁹ evaluations per second ≈ {points / 1e9:.0e} s")

🔍 **What just happened?** One or two parameters are trivial; the 8 regression weights of the prostate model (next
section) would already need $10^{16}$ evaluations — about four months at a billion per second — and 20 parameters are
hopeless. Samplers avoid the grid: they spend their effort only where the posterior has mass.

---
## 5 · Sampling from the posterior: real prostate-cancer data (slides 19–22)

> 📍 **Where we are.** Slides 19–22 introduce *sampling* as the general numerical approach, first in the easy case where
> we can sample the Gaussian posterior directly. We apply it to the real data of Module 06.

The data (Stamey et al. 1989, used on the 02-620 regression slides and in ESL §3.2) are 97 men who underwent radical
prostatectomy. The response is **lpsa**, the log of the prostate-specific antigen level; the eight predictors are log
cancer volume (lcavol), log prostate weight (lweight), age, log benign prostatic hyperplasia amount (lbph), seminal vesicle
invasion (svi, 0/1), log capsular penetration (lcp), Gleason score and the percentage of Gleason scores 4 or 5 (pgg45). We
z-score the predictors and centre lpsa, so no intercept is needed (slide 7's model), and we use all 97 men. For the noise
variance we plug in the least-squares residual variance (Section 6.3 will learn it instead), and we use a weak prior
$\sigma_w^2=1$ on the standardised scale.

**Direct sampling (slides 19–21).** "Draw 100 samples $w^1,\dots,w^{100}$ from $P(w\mid y,X)$; compute any statistic of
interest from the samples." With samples $w^{(1)},\dots,w^{(S)}$, any posterior expectation becomes an average:

$$\mathbb E[f(w)\mid D]\approx\frac1S\sum_{s=1}^S f\big(w^{(s)}\big).$$

*In words:* to know the average of any quantity under the posterior — a weight, the probability that a weight is positive
($f$ = 1 if $w_j>0$, else 0), a prediction for a patient — evaluate it on every sample and average. The error of this
**Monte Carlo** estimate shrinks like $1/\sqrt S$, whatever the number of parameters.

In [ ]:
# ── 5.1 The real prostate data and the exact Gaussian posterior of the 8 regression weights ──
rng = np.random.default_rng(SEED + 5)
pros = load_prostate()
PRED = ["lcavol", "lweight", "age", "lbph", "svi", "lcp", "gleason", "pgg45"]
Xp, _, _ = standardize(pros[PRED].to_numpy(float))          # z-scored predictors (all 97 men)
yp = pros["lpsa"].to_numpy() - pros["lpsa"].mean()          # centred log PSA
Np, Jp = Xp.shape
w_ols, *_ = np.linalg.lstsq(Xp, yp, rcond=None)
SIG_E2_P = np.sum((yp - Xp @ w_ols) ** 2) / (Np - Jp - 1)   # plug-in noise variance (1 dof for the centring)
SIG_W2_P = 1.0                                              # weak prior on standardised weights
mu_p, Sig_p = blr_posterior(Xp, yp, SIG_E2_P, SIG_W2_P)
sd_p = np.sqrt(np.diag(Sig_p))
print(f"{Np} men × {Jp} predictors; plug-in σe² = {SIG_E2_P:.3f}")
print(pd.DataFrame({"OLS": w_ols, "posterior mean": mu_p, "posterior sd": sd_p,
                    "P(w > 0)": stats.norm.cdf(mu_p / sd_p)}, index=PRED).round(3).to_string())

In [ ]:
# ── 5.2 Direct sampling (slide 20): 100 versus 10,000 draws, and statistics computed from them ──
def summarize(samples, names):
    """Posterior summaries computed from samples: mean, sd, 95 % interval, P(w > 0)."""
    return pd.DataFrame({"mean": samples.mean(axis=0), "sd": samples.std(axis=0),
                         "2.5 %": np.percentile(samples, 2.5, axis=0), "97.5 %": np.percentile(samples, 97.5, axis=0),
                         "P(w>0)": (samples > 0).mean(axis=0)}, index=names)


draws100 = sample_mvn(mu_p, Sig_p, 100, rng)
draws10k = sample_mvn(mu_p, Sig_p, 10_000, rng)
exact_tab = pd.DataFrame({"mean": mu_p, "sd": sd_p, "2.5 %": mu_p - 1.96 * sd_p, "97.5 %": mu_p + 1.96 * sd_p,
                          "P(w>0)": stats.norm.cdf(mu_p / sd_p)}, index=PRED)
for label, tab in [("exact", exact_tab), ("100 samples", summarize(draws100, PRED)),
                   ("10,000 samples", summarize(draws10k, PRED))]:
    print(f"— {label} —\n{tab.loc[['lcavol', 'svi', 'gleason']].round(3).to_string()}\n")
print("a derived quantity: P(lcavol effect > svi effect | data) ≈",
      f"{np.mean(draws10k[:, 0] > draws10k[:, 4]):.3f} (10,000 samples)")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
ypos = np.arange(Jp)
axes[0].errorbar(mu_p, ypos, xerr=1.96 * sd_p, fmt="o", color="tab:blue", capsize=3, label="exact 95 % interval")
s100 = summarize(draws100, PRED)
axes[0].plot(s100["2.5 %"], ypos + 0.2, "|", color="tab:orange", ms=8)
axes[0].plot(s100["97.5 %"], ypos + 0.2, "|", color="tab:orange", ms=8, label="from 100 samples")
axes[0].axvline(0, color="0.5", lw=0.8); axes[0].set_yticks(ypos); axes[0].set_yticklabels(PRED)
axes[0].set_xlabel("weight on standardised predictor (lpsa units per SD)"); axes[0].legend(fontsize=7)
axes[0].set_title("real prostate data: posterior of the 8 weights")
grid_w = np.linspace(mu_p[0] - 4 * sd_p[0], mu_p[0] + 4 * sd_p[0], 200)
axes[1].hist(draws10k[:, 0], bins=50, density=True, color="tab:blue", alpha=0.5, label="10,000 samples")
axes[1].hist(draws100[:, 0], bins=15, density=True, histtype="step", color="tab:orange", lw=1.5, label="100 samples")
axes[1].plot(grid_w, stats.norm(mu_p[0], sd_p[0]).pdf(grid_w), color="tab:red", label="exact N(μ, Σ) marginal")
axes[1].set_xlabel("w_lcavol"); axes[1].set_ylabel("density"); axes[1].legend(fontsize=7)
axes[1].set_title("slide 14: a histogram of samples describes the posterior")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With a weak prior and 97 men, the posterior means are close to the OLS estimates, but now each
comes with a full distribution. Log cancer volume has by far the largest effect, and its posterior lies entirely above
zero; Gleason score's 95 % interval straddles zero ($P(w>0)$ near 0.6), so the data do not tell us the sign of its
effect once the other predictors are in the model. Summaries from only 100 samples are already in the right place but
noticeably noisy (compare the orange interval ticks with the exact blue bars); 10,000 samples reproduce the exact
summaries to about two decimals. Derived quantities need no new mathematics: the probability that lcavol's effect exceeds
svi's is just a fraction of samples.

So far we sampled from a posterior that we could also write down. Slide 22 asks the question that motivates the rest of
the lecture: *what if we cannot sample from the joint distribution $P(w\mid D)$, but we can sample from each conditional
distribution $P(w_j\mid w_{-j},D)$* ($w_{-j}$: all elements of $w$ except $w_j$)?

### 📑 Slide connection — `013_Bayesian-MCMC_mz_032725`, slides 16–22

- **What the slides show.** The general recipe — model, prior, posterior (slide 16); the easy case with closed forms by conjugacy
(Gaussian–Gaussian, binomial–Beta, slide 17) and the difficult case where the posterior is described by a histogram of
samples (slide 18); for Bayesian linear regression, draw 100 samples from $\mathcal N(\mu,\Sigma)$ and compute any
statistic from them (slides 19–21); slide 22 asks what to do when we can sample only from the conditionals
$P(w_j\mid w_{-j},Data)$.
- **What this notebook adds.** A table of conjugate pairs linked to earlier modules, the cost of grid approximations, and direct sampling on the
real prostate data with posterior intervals, $P(w_j>0)$, a derived probability and the $1/\sqrt S$ behaviour of
Monte Carlo error (100 versus 10,000 samples).
- **Exam tip.** Remember the Monte Carlo principle: a posterior expectation is approximated by an average over samples, with an
error that shrinks like $1/\sqrt S$ regardless of dimension (for independent samples; MCMC samples count less, Section 7).

---
## 6 · Gibbs sampling (slides 23–35)

> 📍 **Where we are.** Direct sampling needs the joint posterior in a form we can sample. Gibbs sampling needs much
> less: only the ability to sample **one variable at a time, given all the others**.

**The algorithm (slides 23–25, 33).** For a posterior over $w=(w_1,\dots,w_m)$:

- Initialise $w^0$ with random values.
- For $i=1,\dots,S$ (iterations), for $j=1,\dots,m$ (coordinates): sample

$$w_j^{i}\sim p\big(w_j\mid w^{i}_{1:j-1},\;w^{i-1}_{j+1:m},\;\text{Data}\big).$$

*In words:* visit the coordinates one by one; each is redrawn from its distribution given the *most recent* values of all
the others — coordinates $1,\dots,j-1$ were already updated in this sweep, coordinates $j+1,\dots,m$ still hold their
values from the previous sweep.

Slide 24 labels the joint distribution "hard" and the one-variable distribution "easy". (The slide calls the latter a
*marginal* distribution; strictly it is a **full conditional** distribution — the distribution of $w_j$ given everything
else — which is what the formula says.) Slide 25: *if you run Gibbs sampling for many iterations, the samples from later
iterations approximate samples from the target distribution — as if drawn by direct sampling.* Successive samples are,
however, **not independent**: each one starts from the previous one. The sequence $w^1,w^2,\dots$ is a **Markov chain**
(the next state depends only on the current one), and methods that build such chains to sample a distribution are called
**Markov chain Monte Carlo**.

**A first example.** For a standard bivariate Gaussian with correlation $\rho$, the full conditionals are 1-D Gaussians:

$$x_1\mid x_2\sim\mathcal N\big(\rho\,x_2,\;1-\rho^2\big),\qquad x_2\mid x_1\sim\mathcal N\big(\rho\,x_1,\;1-\rho^2\big).$$

*In words:* given $x_2$, the best guess for $x_1$ is $\rho x_2$, and the remaining uncertainty is smaller the stronger the
correlation (this is the conditional-Gaussian formula of Module 18 with unit variances).

In [ ]:
# ── 6.1 Gibbs sampling a bivariate Gaussian: weak versus strong correlation ──
rng = np.random.default_rng(SEED + 6)


def gibbs_bivariate(rho: float, n_iter: int, rng, start=(-2.5, 2.5)):
    """Gibbs sampler for (x1, x2) ~ N(0, [[1, ρ], [ρ, 1]]); returns the samples and the zig-zag path."""
    x1, x2 = start
    sd = np.sqrt(1 - rho ** 2)
    samples, path = np.empty((n_iter, 2)), [(x1, x2)]
    for t in range(n_iter):
        x1 = rho * x2 + sd * rng.standard_normal()      # x1 | x2
        path.append((x1, x2))
        x2 = rho * x1 + sd * rng.standard_normal()      # x2 | x1 (uses the new x1)
        path.append((x1, x2))
        samples[t] = x1, x2
    return samples, np.array(path)


def lag_autocorr(x, lag: int = 1) -> float:
    """Correlation between a chain and itself shifted by `lag` steps."""
    x = x - x.mean()
    return float(np.dot(x[:-lag], x[lag:]) / np.dot(x, x))


g = np.linspace(-3.2, 3.2, 120)
gx, gy = np.meshgrid(g, g)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
for ax, rho in zip(axes, [0.5, 0.98]):
    smp, path = gibbs_bivariate(rho, 3000, rng)
    dens = np.exp(-(gx ** 2 - 2 * rho * gx * gy + gy ** 2) / (2 * (1 - rho ** 2)))
    ax.contour(gx, gy, dens, levels=[0.05, 0.3, 0.7], colors="0.6", linewidths=0.8)
    ax.scatter(smp[100:, 0], smp[100:, 1], s=2, color="tab:blue", alpha=0.25)
    ax.plot(path[:41, 0], path[:41, 1], "-o", color="tab:red", ms=2.5, lw=1, label="first 20 iterations")
    ax.set_title(f"ρ = {rho}: sample corr {np.corrcoef(smp[100:].T)[0, 1]:.3f}, "
                 f"lag-1 autocorr of x1 {lag_autocorr(smp[100:, 0]):.3f} (ρ² = {rho ** 2:.3f})")
    ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$"); ax.set_aspect("equal"); ax.legend(fontsize=7, loc="lower right")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Gibbs moves only parallel to the axes, producing the red staircase. With $\rho=0.5$ the
staircase quickly reaches the bulk of the distribution and the samples (blue) fill the ellipse. With $\rho=0.98$ each step
is tiny — the conditional standard deviation is $\sqrt{1-0.98^2}=0.2$ — so the chain crawls along the narrow ridge and
successive samples are nearly identical. For this target the lag-1 autocorrelation of $x_1$ is exactly $\rho^2$ in theory,
and the measured values match (0.253 and 0.968 versus 0.25 and 0.96). **Strong posterior correlation = slow mixing** is the main weakness of
Gibbs sampling; remember the linked SNPs of 3.3.

### 6.2 Gibbs sampling for Bayesian linear regression

We need the full conditional of one weight. Write $r_j=y-\sum_{k\neq j}x_kw_k$ for the **partial residual** — the part of
$y$ not explained by the other predictors ($x_k$ is column $k$ of $X$). As a function of $w_j$ alone the log-posterior is

$$\log p(w_j\mid w_{-j},y)=-\frac{\|r_j-x_jw_j\|^2}{2\sigma_e^2}-\frac{w_j^2}{2\sigma_w^2}+\text{const}
=-\tfrac12\Big(\frac{x_j^\top x_j}{\sigma_e^2}+\frac{1}{\sigma_w^2}\Big)w_j^2+w_j\frac{x_j^\top r_j}{\sigma_e^2}+\text{const},$$

*In words:* a one-dimensional Gaussian in $w_j$ — the same completing-the-square pattern as in Section 3 — so

$$w_j\mid w_{-j},y\;\sim\;\mathcal N\big(m_j,\;1/\tau_j\big),\qquad \tau_j=\frac{x_j^\top x_j}{\sigma_e^2}+\frac{1}{\sigma_w^2},
\qquad m_j=\frac{x_j^\top r_j/\sigma_e^2}{\tau_j}.$$

*In words:* each weight is redrawn around the value that a one-predictor ridge regression of the current partial residual
on $x_j$ would give, with a spread set by how much information that predictor carries.

In [ ]:
# ── 6.2 Gibbs sampler for Bayesian linear regression (known σe²), checked against the closed form ──
def gibbs_blr(X, y, sigma_e2, sigma_w2, n_iter: int, rng, w0=None):
    """Coordinate-wise Gibbs sampler for the posterior of slide 15; returns an (n_iter × J) array of samples."""
    N, J = X.shape
    w = np.zeros(J) if w0 is None else np.array(w0, float)
    r = y - X @ w                                          # current residual, updated in place
    xx = np.sum(X ** 2, axis=0)                            # x_jᵀ x_j for every predictor
    out = np.empty((n_iter, J))
    for t in range(n_iter):
        for j in range(J):
            r += X[:, j] * w[j]                            # partial residual r_j (remove predictor j's contribution)
            tau = xx[j] / sigma_e2 + 1.0 / sigma_w2        # conditional precision
            m = (X[:, j] @ r / sigma_e2) / tau             # conditional mean
            w[j] = m + rng.standard_normal() / np.sqrt(tau)
            r -= X[:, j] * w[j]                            # put the new contribution back
        out[t] = w
    return out


BURN = 500
chain = gibbs_blr(Xp, yp, SIG_E2_P, SIG_W2_P, 5500, rng)
kept = chain[BURN:]
print(f"{len(kept)} Gibbs samples after discarding {BURN} burn-in iterations")
print(f"max |Gibbs mean − μ| = {np.abs(kept.mean(axis=0) - mu_p).max():.4f}   (posterior sds are {sd_p.min():.3f}–{sd_p.max():.3f})")
print(f"max |Gibbs covariance − Σ| = {np.abs(np.cov(kept.T) - Sig_p).max():.5f}   (largest |Σ| entry {np.abs(Sig_p).max():.4f})")
print(pd.DataFrame({"exact mean": mu_p, "Gibbs mean": kept.mean(axis=0), "exact sd": sd_p,
                    "Gibbs sd": kept.std(axis=0)}, index=PRED).round(3).to_string())
fig, axes = plt.subplots(1, 4, figsize=(12, 2.9))
for ax, name in zip(axes, ["lcavol", "svi", "lcp", "pgg45"]):
    j = PRED.index(name)
    gw = np.linspace(mu_p[j] - 4 * sd_p[j], mu_p[j] + 4 * sd_p[j], 200)
    ax.hist(kept[:, j], bins=40, density=True, color="tab:blue", alpha=0.5, label="Gibbs samples")
    ax.plot(gw, stats.norm(mu_p[j], sd_p[j]).pdf(gw), color="tab:red", label="exact")
    ax.set_xlabel(f"w_{name}"); ax.set_title(f"{name}")
axes[0].set_ylabel("density"); axes[0].legend(fontsize=7)
fig.suptitle("real prostate data: Gibbs samples versus the closed-form posterior", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Without ever inverting a matrix, the Gibbs sampler reproduces the closed-form posterior of
slide 15: the sample means agree with $\mu$ to within a small fraction of a posterior standard deviation, the sample
covariance matches $\Sigma$, and the histograms follow the exact Gaussian marginals. This is the promise of slide 25 —
and the properties of slide 35: once the chain has stabilised, the samples approximate the joint distribution; the
marginal of any subset is obtained by simply looking at that subset's columns; and expectations are averages.

### 6.3 Learning the noise variance too: a second conditional

So far $\sigma_e^2$ was plugged in. In a fully Bayesian treatment it is unknown and gets a prior. The conjugate choice is
the **inverse-gamma** distribution, $\sigma_e^2\sim\text{IG}(a_0,b_0)$, whose density is
$\propto(\sigma_e^2)^{-a_0-1}e^{-b_0/\sigma_e^2}$. Multiplying by the Gaussian likelihood,
$\propto(\sigma_e^2)^{-N/2}\exp\{-\|y-Xw\|^2/(2\sigma_e^2)\}$, gives the full conditional

$$\sigma_e^2\mid w,y\;\sim\;\text{IG}\Big(a_0+\frac N2,\;\;b_0+\frac{\|y-Xw\|^2}{2}\Big).$$

*In words:* the shape parameter counts the data points and the rate parameter adds half the residual sum of squares; the
noise variance is redrawn around "residual sum of squares divided by $N$". The joint posterior of $(w,\sigma_e^2)$ is no
longer a simple Gaussian, but both full conditionals are easy — so Gibbs simply alternates between them. (To draw from
IG$(a,b)$, draw $g$ from a Gamma$(a,\text{rate }b)$ distribution and take $1/g$.)

In [ ]:
# ── 6.3 Gibbs with an unknown noise variance (inverse-gamma prior) ──
def gibbs_blr_unknown_noise(X, y, sigma_w2, a0, b0, n_iter: int, rng):
    """Gibbs sampler for (w, σe²): w_j | rest is Gaussian (6.2), σe² | w is IG(a0 + N/2, b0 + RSS/2)."""
    N, J = X.shape
    w, s2 = np.zeros(J), np.var(y)
    r = y.copy()
    xx = np.sum(X ** 2, axis=0)
    W_out, s2_out = np.empty((n_iter, J)), np.empty(n_iter)
    for t in range(n_iter):
        for j in range(J):
            r += X[:, j] * w[j]
            tau = xx[j] / s2 + 1.0 / sigma_w2
            w[j] = (X[:, j] @ r / s2) / tau + rng.standard_normal() / np.sqrt(tau)
            r -= X[:, j] * w[j]
        s2 = 1.0 / rng.gamma(a0 + N / 2, 1.0 / (b0 + r @ r / 2))   # NumPy's gamma takes shape and scale = 1/rate
        W_out[t], s2_out[t] = w, s2
    return W_out, s2_out


W_u, s2_u = gibbs_blr_unknown_noise(Xp, yp, SIG_W2_P, a0=1.0, b0=1.0, n_iter=5500, rng=rng)
W_u, s2_u = W_u[BURN:], s2_u[BURN:]
print(f"σe²: posterior mean {s2_u.mean():.3f}, 95 % interval [{np.percentile(s2_u, 2.5):.3f}, "
      f"{np.percentile(s2_u, 97.5):.3f}]; least-squares plug-in {SIG_E2_P:.3f}")
print("posterior sd of the weights, σe² known vs learned:")
print(pd.DataFrame({"σe² plugged in": kept.std(axis=0), "σe² learned": W_u.std(axis=0)}, index=PRED).round(3).T.to_string())
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].plot(s2_u[:500], lw=0.7, color="tab:blue")
axes[0].set_xlabel("iteration (after burn-in)"); axes[0].set_ylabel("σe²"); axes[0].set_title("trace of σe²")
axes[1].hist(s2_u, bins=50, density=True, color="tab:blue", alpha=0.6)
axes[1].axvline(SIG_E2_P, color="tab:red", ls="--", label="least-squares plug-in")
axes[1].set_xlabel("σe² (lpsa² units)"); axes[1].set_ylabel("density"); axes[1].legend(fontsize=7)
axes[1].set_title("real prostate data: posterior of the noise variance")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The sampler learned the noise variance from the data: its posterior is centred near the
least-squares value and has a visible width — 97 men do not determine $\sigma_e^2$ exactly. Because that uncertainty is
now propagated, the weights' posterior standard deviations change slightly compared with plugging in a single value
(printed table). Extending a Gibbs sampler to a new unknown only requires one more full conditional — this modularity is
why Gibbs sampling became the workhorse of applied Bayesian statistics.

✏️ **Your turn.** (a) In 6.2, replace `SIG_W2_P` by 0.01 (a strong prior). How do the Gibbs means and the closed-form
$\mu$ change — and do they still agree? (b) In 6.3, try the prior `a0 = 50, b0 = 5` (prior mean $b_0/(a_0-1)\approx0.1$).
How far does the posterior of $\sigma_e^2$ move?

### 📑 Slide connection — `013_Bayesian-MCMC_mz_032725`, slides 23–35

- **What the slides show.** Gibbs sampling next to direct sampling: initialise $w^0$; for each iteration $i$ and coordinate $j$ sample
$w_j^i$ from $p(w_j\mid w^i_{1:j-1},w^{i-1}_{j+1:m},Data)$ (slides 23, 33; slide 34 is hidden); sampling from the joint is
"hard", from the one-variable distribution "easy" (slide 24); after many iterations the samples approximate draws from
the target (slide 25); properties: initial samples are bad, later samples approximate the joint, marginals by ignoring
other variables, expectations by averaging (slide 35). Slides 26–32 (hidden) repeat 19–25.
- **What this notebook adds.** Derivations of the full conditionals for Bayesian linear regression and for the noise variance
(inverse-gamma), three from-scratch samplers, a bivariate demonstration that strong correlation means slow mixing
(lag-1 autocorrelation $=\rho^2$, verified), and a check that Gibbs reproduces the closed-form $\mu$ and $\Sigma$ on real
data. Terminology: slide 24 calls $p(w_j\mid w_{-j},Data)$ a "marginal" distribution; it is a **full conditional**.
- **Exam tip.** Be able to write the Gibbs loop with the correct superscripts (new values for $1{:}j-1$, old values for
$j+1{:}m$) and to derive a full conditional by keeping only the factors that contain the variable being sampled.

---
## 7 · Why Gibbs works, and how to know when to trust it (slides 36–43)

> 📍 **Where we are.** Gibbs reproduced known answers. Slide 36 (hidden in the PDF) asks: *simple algorithm, but why
> does it work — what is the intuition?* — and slides 39–43 ask the practical questions: how many iterations to discard,
> and how many to keep?

**Intuition: coordinate descent (slides 37–38, hidden in the PDF).** Gradient descent ([Module 04](04_calculus_and_optimization.ipynb),
Section 4) moves all parameters at once. **Coordinate descent** (Module 04, Section 8; the lasso solver of
[Module 07](07_ridge_lasso_sparse_regression.ipynb), Section 7) — strictly *ascent* here, since we maximise — updates one
parameter at a time, holding the others fixed. Slide 38 writes the sweep as "max $\log p(w_1\mid w_{-1},y,X)$, max
$\log p(w_2\mid w_{-2},y,X)$, …, max $\log p(w_J\mid w_{-J},y,X)$", i.e.
$w_j\leftarrow\arg\max_{w_j}\log p(w_j\mid w_{-j},y,X)$. Put it next to the Gibbs sweep of slide 33 ("sample from
$p(w_1\mid w_{2:m}^{i-1},y,X)$, …") and the analogy is plain: Gibbs makes the same moves, except that it *samples* $w_j$
from $p(w_j\mid w_{-j},y,X)$ instead of jumping to its maximum. Coordinate ascent climbs to the peak of
the posterior (the MAP); Gibbs explores the whole hill in proportion to its height. (A precise justification: each Gibbs
update leaves the target distribution unchanged, so once the chain has reached it, it stays there — the target is the
chain's **stationary distribution**.)

For a Gaussian posterior $\mathcal N(\mu,\Sigma)$ with precision $Q=\Sigma^{-1}$, the full conditional of coordinate $j$
is (Module 18, Section 7):

$$w_j\mid w_{-j}\sim\mathcal N\Big(\mu_j-\frac{1}{Q_{jj}}\sum_{k\neq j}Q_{jk}(w_k-\mu_k),\;\frac{1}{Q_{jj}}\Big).$$

*In words:* coordinate ascent sets $w_j$ to the conditional mean; Gibbs adds Gaussian noise with variance $1/Q_{jj}$.

In [ ]:
# ── 7.1 Coordinate ascent versus Gibbs on the linked-SNP posterior of 3.3 ──
rng = np.random.default_rng(SEED + 7)
Q2 = np.linalg.inv(S2)                                         # posterior precision of (w_A, w_B)


def coordinate_sweeps(Q, mu, start, n_sweeps: int, rng=None):
    """Update one coordinate at a time: set it to its conditional mean (rng=None) or sample it (Gibbs)."""
    w = np.array(start, float)
    path = [w.copy()]
    for _ in range(n_sweeps):
        for j in range(len(w)):
            others = np.arange(len(w)) != j
            m = mu[j] - Q[j, others] @ (w[others] - mu[others]) / Q[j, j]     # conditional mean
            w[j] = m if rng is None else m + rng.standard_normal() / np.sqrt(Q[j, j])
            path.append(w.copy())
    return np.array(path)


start = (-0.4, 0.9)
path_ca = coordinate_sweeps(Q2, mu2, start, 25)
path_gibbs = coordinate_sweeps(Q2, mu2, start, 25, rng)
print(f"coordinate ascent after 25 sweeps: {np.round(path_ca[-1], 4)}   posterior mean (= MAP): {np.round(mu2, 4)}")
fig, ax = plt.subplots(figsize=(5.4, 4.4))
ax.contour(ga, gb, dens_ld, levels=[0.05, 0.25, 0.6, 0.9], colors="0.6", linewidths=0.8)
ax.plot(path_ca[:, 0], path_ca[:, 1], "-o", color="tab:green", ms=3, lw=1.2, label="coordinate ascent (maximise)")
ax.plot(path_gibbs[:, 0], path_gibbs[:, 1], "-o", color="tab:red", ms=3, lw=0.8, alpha=0.8, label="Gibbs (sample)")
ax.plot(*start, "ks", ms=7, label="common start")
ax.set_xlabel("w_A"); ax.set_ylabel("w_B"); ax.legend(fontsize=7)
ax.set_title("same moves, different purpose:\nclimb to the peak versus explore the hill")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Both methods move parallel to the axes from the same starting point. Coordinate ascent (green)
zig-zags along the narrow ridge towards the posterior mean — which, for a Gaussian, is also the MAP — and after 25
sweeps is within 0.001 of it. Gibbs (red)
makes similar moves, but with random steps it keeps wandering over the whole ellipse instead of stopping. Both are slow
along the tilted ridge for the same reason: one-coordinate moves are short when the coordinates are strongly correlated.

### 7.2 Burn-in: "initial samples are bad" (slides 39–41)

Slide 39 splits the chain into initial iterations (**burn-in**, discarded) and later iterations, which are treated as
samples from $p(w\mid y,X)$. How many to discard? Too few, and the samples still remember the arbitrary starting point;
too many only wastes computation — "wait until the distribution stabilises" (slide 41). A **trace plot** (value versus
iteration) shows it.

In [ ]:
# ── 7.2 Burn-in on the real prostate posterior: start far away and watch the chain settle ──
far_chain = gibbs_blr(Xp, yp, SIG_E2_P, SIG_W2_P, 3000, rng, w0=np.full(Jp, 5.0))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for name, col in [("lcavol", "tab:blue"), ("pgg45", "tab:orange")]:
    j = PRED.index(name)
    axes[0].plot(far_chain[:100, j], color=col, lw=1, label=f"w_{name}")
    axes[0].axhspan(mu_p[j] - 2 * sd_p[j], mu_p[j] + 2 * sd_p[j], color=col, alpha=0.15)
axes[0].axvspan(0, 20, color="0.85", zorder=0)
axes[0].text(1, axes[0].get_ylim()[1] * 0.9, "burn-in", fontsize=8)
axes[0].set_xlabel("iteration"); axes[0].set_ylabel("sampled weight"); axes[0].legend(fontsize=7)
axes[0].set_title("trace plot from the start w⁰ = 5 (bands: exact mean ± 2 sd)")
j = PRED.index("lcavol")
steps = np.arange(1, len(far_chain) + 1)
axes[1].plot(steps, np.cumsum(far_chain[:, j]) / steps, color="tab:red", label="keeping every sample")
burn = 50
axes[1].plot(steps[burn:], np.cumsum(far_chain[burn:, j]) / np.arange(1, len(far_chain) - burn + 1),
             color="tab:blue", label=f"after discarding {burn} burn-in samples")
axes[1].axhline(mu_p[j], color="k", ls="--", lw=1, label="exact posterior mean")
axes[1].set_xscale("log"); axes[1].set_xlabel("number of iterations (log scale)"); axes[1].set_ylabel("running mean of w_lcavol")
axes[1].set_ylim(mu_p[j] - 0.5, mu_p[j] + 0.5); axes[1].legend(fontsize=7)
axes[1].set_title("running averages: wait until estimates stabilise")
plt.tight_layout(); plt.show()
for b in (0, 20, 50):
    print(f"discard {b:2d}: mean of w_lcavol over 3,000 iterations = {far_chain[b:, j].mean():.4f} (exact {mu_p[j]:.4f})")

🔍 **What just happened?** Started at the absurd value $w=5$ for every predictor, the chain is thrown far off at first
(the first draw of $w_{\text{lcavol}}$ is about −12, because the other seven weights are still at 5) and then reaches
the region where the posterior lives (shaded bands) within a few iterations for lcavol and about ten for pgg45 — a
predictor strongly correlated with others. The running average that keeps the burn-in samples is dragged by the starting
point and recovers only slowly; discarding the first 20 or 50 iterations halves the gap to the exact mean (0.008 → 0.004),
and the remaining difference of about 0.004 is ordinary Monte Carlo error (compare the errors computed in 7.3). With well-informed data this burn-in is short;
for multimodal or strongly correlated posteriors it can be very long.

### 7.3 How many samples? Autocorrelation and effective sample size (slides 42–43)

Consecutive Gibbs samples are correlated, so $S$ MCMC samples carry less information than $S$ independent ones. The
**autocorrelation** $\rho_\ell$ at lag $\ell$ is the correlation between the chain and itself shifted by $\ell$ steps; the
**effective sample size** is

$$S_{\text{eff}}=\frac{S}{1+2\sum_{\ell\geq1}\rho_\ell}.$$

*In words:* if each sample is strongly correlated with its neighbours, the chain behaves like a much shorter sequence of
independent draws; $S_{\text{eff}}$ is that equivalent number. (In practice the sum is cut off once $\rho_\ell$ becomes
negligible.) Slide 43's advice — draw samples "until the estimates stabilise" — amounts to making $S_{\text{eff}}$ large
enough for the precision you need: the Monte Carlo error of a posterior mean is about $\text{sd}/\sqrt{S_{\text{eff}}}$.

In [ ]:
# ── 7.3 Autocorrelation and effective sample size of the prostate Gibbs chain (from 6.2) ──
def autocorr(x, max_lag: int) -> np.ndarray:
    """Autocorrelations ρ_0 … ρ_max_lag of a chain."""
    x = x - x.mean()
    v = np.dot(x, x)
    return np.array([1.0] + [np.dot(x[:-l], x[l:]) / v for l in range(1, max_lag + 1)])


def effective_sample_size(x, max_lag: int = 200) -> float:
    """S / (1 + 2 Σ ρ_ℓ), summing ρ_ℓ until it first drops below 0.05."""
    rho = autocorr(x, max_lag)[1:]
    cut = np.argmax(rho < 0.05) if np.any(rho < 0.05) else len(rho)
    return len(x) / (1 + 2 * rho[:cut].sum())


ess = np.array([effective_sample_size(kept[:, j]) for j in range(Jp)])
print(pd.DataFrame({"lag-1 autocorr": [autocorr(kept[:, j], 1)[1] for j in range(Jp)], "ESS (of 5,000)": ess.astype(int),
                    "MC error of mean": kept.std(axis=0) / np.sqrt(ess)}, index=PRED).round(3).to_string())
biv, _ = gibbs_bivariate(0.98, 5000, rng)
print(f"\nbivariate Gaussian with ρ = 0.98: ESS of x1 = {effective_sample_size(biv[100:, 0]):.0f} of {len(biv) - 100}")
fig, ax = plt.subplots(figsize=(6.5, 3.3))
for series, lab in [(kept[:, PRED.index("lweight")], "prostate w_lweight"), (kept[:, PRED.index("pgg45")], "prostate w_pgg45"),
                    (biv[100:, 0], "bivariate Gaussian, ρ = 0.98")]:
    ax.plot(autocorr(series, 60), label=lab)
ax.axhline(0, color="0.5", lw=0.8); ax.set_xlabel("lag ℓ (iterations)"); ax.set_ylabel("autocorrelation ρ_ℓ")
ax.set_title("how quickly does the chain forget where it was?"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** For the prostate posterior the weights of lcp, gleason and pgg45 — predictors that are
correlated with one another — have the highest autocorrelation and the smallest effective sample sizes, but every weight
still has over a thousand effective samples out of 5,000, which makes the Monte Carlo error of each posterior mean tiny
compared with its posterior standard deviation. The strongly correlated bivariate Gaussian is a different story: its
autocorrelation decays very slowly, and 4,900 iterations are worth only about a hundred independent draws.

### 7.4 Several chains from different starts

A single chain cannot tell you whether it has explored the whole posterior. A standard check runs several chains from
widely dispersed starting points and compares them with the **Gelman–Rubin statistic** $\hat R$: the square root of the
ratio of the total variance (between and within chains) to the average within-chain variance. If the chains have all
found the same distribution, the between-chain part is negligible and $\hat R\approx1$;
values above about 1.01–1.1 mean "run longer" (or that something is wrong).

In [ ]:
# ── 7.4 Four chains from dispersed starts, and the Gelman–Rubin R̂ ──
def gelman_rubin(chains: np.ndarray) -> float:
    """R̂ for one parameter from an (m chains × n draws) array."""
    m, n = chains.shape
    B = n * chains.mean(axis=1).var(ddof=1)                # between-chain variance
    W = chains.var(axis=1, ddof=1).mean()                  # average within-chain variance
    var_hat = (n - 1) / n * W + B / n
    return float(np.sqrt(var_hat / W))


starts = rng.normal(0, 5, size=(4, Jp))
runs = np.stack([gibbs_blr(Xp, yp, SIG_E2_P, SIG_W2_P, 1500, rng, w0=s) for s in starts])   # 4 × 1500 × J
rhat = [gelman_rubin(runs[:, 500:, j]) for j in range(Jp)]
print("prostate posterior, 4 chains, iterations 500–1500:")
print(pd.DataFrame({"R̂": rhat}, index=PRED).round(3).T.to_string())
print("\nslowly mixing target (bivariate Gaussian, ρ = 0.999), 4 chains started at ±2 and ±3 along the ridge:")
for n_it in (300, 1000, 5000):
    ch = np.stack([gibbs_bivariate(0.999, n_it, rng, start=(s0, s0))[0][:, 0] for s0 in (-3, -2, 2, 3)])
    print(f"   {n_it:5d} iterations (first 20 % discarded): R̂ of x1 = {gelman_rubin(ch[:, n_it // 5:]):.3f}")

🔍 **What just happened?** For the prostate posterior, all four chains agree after burn-in ($\hat R$ within a percent of
1 for every weight). For the strongly correlated bivariate target, chains started at different points along the ridge
are still far apart after 300 iterations ($\hat R$ far above 1), and even after 1,000 or 5,000 iterations $\hat R$
stays above the usual 1.01–1.1 thresholds: these chains must be run much longer, or the sampler improved (see ✏️ below). Diagnostics like these can show that a chain has *not* converged; they can
never prove that it has, for instance if all chains are stuck in the same mode.

✏️ **Your turn.** Run `gibbs_bivariate(0.999, …)` and compute the effective sample size of 5,000 iterations. Then
reparameterise: Gibbs-sample the *independent* coordinates $u=(x_1+x_2)/\sqrt2$, $v=(x_1-x_2)/\sqrt2$ instead. Why does a
rotation make the sampler so much better — and why is that usually impossible in real, high-dimensional models?

### 📑 Slide connection — `013_Bayesian-MCMC_mz_032725`, slides 36–43

- **What the slides show.** "Simple algorithm, but why does it work? What is the intuition?" (slide 36); coordinate descent — in each
iteration maximise for one variable, fixing all others — contrasted with gradient descent (slide 37), and the coordinate
sweep "max $\log p(w_j\mid w_{-j},y,X)$" for $j=1,\dots,J$, laid out like the Gibbs sweep of slide 33 (slide 38; 36–38
are hidden in the PDF); the burn-in diagram (slide 39); how many burn-in iterations ("wait until the distribution
stabilises") and how many samples ("wait until the estimates stabilise") (slides 40–43).
- **What this notebook adds.** A side-by-side run of coordinate ascent (converges to the MAP) and Gibbs (explores the posterior), the notion of
a stationary distribution, trace plots and running averages from a deliberately bad start on real data, autocorrelation
and effective sample size, and multiple chains with the Gelman–Rubin $\hat R$.
- **Exam tip.** Typical exam question: "why discard burn-in?" (the chain has not yet forgotten its arbitrary start) and "why
are 5,000 Gibbs samples not worth 5,000 independent samples?" (autocorrelation; quote the effective sample size).

---
## 8 · Beyond Gibbs: Metropolis–Hastings (slides 44–45)

> 📍 **Where we are.** Gibbs needs full conditionals we can sample. Slide 44: *for even more complex models, sampling
> from this conditional distribution is not possible; then, other MCMC methods.* The most important one is
> **Metropolis–Hastings (MH)**, which can update many dimensions at once (slide 45).

**Example: a risk allele in a case–control study.** Logistic regression (Module 09) models the probability of disease
given genotype $g\in\{0,1,2\}$ as $P(y=1\mid g)=\sigma(w_0+w_1g)$. With a Gaussian prior on $(w_0,w_1)$ the posterior

$$p(w\mid D)\;\propto\;\prod_{i=1}^N\sigma(z_i)^{y_i}\big(1-\sigma(z_i)\big)^{1-y_i}\cdot\mathcal N(w;0,\,s^2I),\qquad z_i=w_0+w_1g_i,$$

*In words:* the likelihood of every person's disease status times the prior — a perfectly well-defined distribution that,
unlike the Gaussian case, is **not** any named distribution, and whose full conditionals are not either. But we can always
*evaluate* it up to the unknown constant $P(D)$. Using $\log\sigma(z)=z-\log(1+e^z)$ and $\log(1-\sigma(z))=-\log(1+e^z)$,
the log-posterior is $\sum_i\big[y_iz_i-\log(1+e^{z_i})\big]-\|w\|^2/(2s^2)+\text{const}$.

**Random-walk Metropolis–Hastings.** From the current $w$:
1. **Propose** $w'=w+\varepsilon$ with $\varepsilon\sim\mathcal N(0,\text{step}^2I)$ — all coordinates at once.
2. **Accept** $w'$ with probability $\min\!\big(1,\;p(w'\mid D)/p(w\mid D)\big)$; otherwise stay at $w$ (and record $w$
   again).

$$\alpha(w\to w')=\min\Big(1,\frac{p(w'\mid D)\,q(w\mid w')}{p(w\mid D)\,q(w'\mid w)}\Big).$$

*In words:* uphill moves are always accepted, downhill moves sometimes — the more downhill, the less often — so the chain
spends time in each region in proportion to its posterior probability. The unknown constant $P(D)$ **cancels** in the
ratio, which is why MH works for any posterior we can evaluate. The general MH ratio includes the proposal densities $q$;
for a symmetric proposal like our Gaussian random walk they cancel (the original Metropolis algorithm). Slide 44 phrases
the idea as "approximate the conditional posterior with a tractable distribution (Gaussian, uniform) to draw samples": the
tractable distribution is the **proposal**, and the accept/reject step corrects for the fact that it is not the posterior,
so the chain still targets the exact posterior.

In [ ]:
# ── 8.1 A SIMULATED case–control study and its (non-conjugate) logistic-regression posterior ──
rng = np.random.default_rng(SEED + 8)
Gcc, _, _ = simulate_genotypes(rng, 400, 1, maf_range=(0.3, 0.3))
g_cc = Gcc[:, 0].astype(float)
W_TRUE = np.array([-1.0, 0.5])                            # baseline log-odds, log odds ratio per allele
y_cc = rng.binomial(1, sigmoid(W_TRUE[0] + W_TRUE[1] * g_cc))
PRIOR_SD = 2.0
print(f"{len(y_cc)} individuals, {y_cc.sum()} cases; cases by genotype 0/1/2:",
      [f"{y_cc[g_cc == v].sum()}/{np.sum(g_cc == v)}" for v in (0, 1, 2)])


def log_post_logistic(w) -> float:
    """Unnormalised log-posterior: Σ [y z − log(1 + e^z)] − ‖w‖²/(2 s²), with z = w0 + w1·g."""
    z = w[0] + w[1] * g_cc
    return float(np.sum(y_cc * z - np.logaddexp(0.0, z)) - np.sum(np.square(w)) / (2 * PRIOR_SD ** 2))


# Reference answer by brute force on a 2-D grid (possible here only because there are just 2 parameters)
w0_axis, w1_axis = np.linspace(-2.0, 0.0, 201), np.linspace(-0.4, 1.4, 201)
W0g, W1g = np.meshgrid(w0_axis, w1_axis, indexing="ij")
Zg = W0g[..., None] + W1g[..., None] * g_cc                 # 201 × 201 × 400 linear predictors
logp_grid = np.sum(y_cc * Zg - np.logaddexp(0.0, Zg), axis=-1) - (W0g ** 2 + W1g ** 2) / (2 * PRIOR_SD ** 2)
p_grid = np.exp(logp_grid - logp_grid.max())
p_grid /= p_grid.sum()
grid_mean_w1 = np.sum(p_grid * W1g)
grid_sd_w1 = np.sqrt(np.sum(p_grid * (W1g - grid_mean_w1) ** 2))
print(f"grid posterior of w1 (log odds ratio): mean {grid_mean_w1:.4f}, sd {grid_sd_w1:.4f}  (true value {W_TRUE[1]})")

In [ ]:
# ── 8.2 Random-walk Metropolis–Hastings and the choice of proposal width ──
def metropolis(log_post, w_start, step: float, n_iter: int, rng):
    """Random-walk Metropolis: propose w' = w + step·N(0, I) (all coordinates at once); accept with prob min(1, p(w')/p(w))."""
    w = np.array(w_start, float)
    lp = log_post(w)
    out, n_acc = np.empty((n_iter, len(w))), 0
    for t in range(n_iter):
        proposal = w + step * rng.standard_normal(len(w))
        lp_prop = log_post(proposal)
        if np.log(rng.random()) < lp_prop - lp:               # accept with probability min(1, ratio)
            w, lp = proposal, lp_prop
            n_acc += 1
        out[t] = w                                            # a rejected proposal repeats the current state
    return out, n_acc / n_iter


runs_mh = {}
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
for ax, step in zip(axes, [0.01, 0.15, 1.5]):
    smp, acc = metropolis(log_post_logistic, [0.0, 0.0], step, 10_000, rng)
    runs_mh[step] = smp[1000:]
    ess_mh = effective_sample_size(smp[1000:, 1], max_lag=2000)
    print(f"step {step:5.2f}: acceptance rate {acc:.2f}, posterior mean of w1 {smp[1000:, 1].mean():.4f}, "
          f"sd {smp[1000:, 1].std():.4f}, ESS {ess_mh:.0f} of 9,000")
    ax.plot(smp[:3000, 1], lw=0.6, color="tab:blue")
    ax.axhline(grid_mean_w1, color="tab:red", ls="--", lw=1)
    ax.set_title(f"step = {step}: acceptance {acc:.0%}"); ax.set_xlabel("iteration")
axes[0].set_ylabel("w1 (log odds ratio per allele)")
fig.suptitle("SIMULATED case–control data: Metropolis–Hastings traces (red: grid posterior mean)", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The proposal width controls everything. With tiny steps (0.01) almost every proposal is
accepted, but the chain creeps: it needs more than a thousand iterations to drift from the start $(0,0)$ to the bulk of
the posterior, its samples are extremely autocorrelated (effective sample size in single digits), and even after the
1,000 burn-in iterations its mean is still biased. With huge steps (1.5) almost every proposal lands in a region of negligible posterior
probability and is rejected, so the chain sits still for long stretches (flat segments). The intermediate width (0.15)
accepts a moderate fraction of proposals and explores efficiently: its posterior mean and standard deviation match the
grid reference, and its effective sample size is the largest. (For random-walk proposals, acceptance rates of roughly
20–50 % are a common rule of thumb.)

Slide 47 calls Gibbs sampling "a special case of MCMC". More precisely, it is a special case of MH: proposing $w_j$ from
its exact full conditional gives an acceptance probability of exactly 1, so no proposal is ever rejected.

In [ ]:
# ── 8.3 The quantity a geneticist wants: the posterior of the odds ratio per allele ──
w_mh = runs_mh[0.15]
odds_ratio = np.exp(w_mh[:, 1])                            # a derived quantity: transform every sample
lo, hi = np.percentile(odds_ratio, [2.5, 97.5])
print(f"odds ratio per allele: posterior median {np.median(odds_ratio):.2f}, 95 % credible interval [{lo:.2f}, {hi:.2f}] "
      f"(true {np.exp(W_TRUE[1]):.2f})")
print(f"P(odds ratio > 1 | data) = {np.mean(odds_ratio > 1):.4f}")
print(f"MH versus grid — mean of w1: {w_mh[:, 1].mean():.4f} vs {grid_mean_w1:.4f}; sd: {w_mh[:, 1].std():.4f} vs {grid_sd_w1:.4f}")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].contour(W0g, W1g, p_grid, levels=5, colors="tab:red", linewidths=0.8)
axes[0].scatter(w_mh[::10, 0], w_mh[::10, 1], s=3, alpha=0.3, color="tab:blue")
axes[0].set_xlabel("w0 (baseline log-odds)"); axes[0].set_ylabel("w1 (log odds ratio)")
axes[0].set_title("MH samples (blue) on the grid posterior (red contours)")
axes[1].hist(odds_ratio, bins=50, density=True, color="tab:blue", alpha=0.6)
axes[1].axvline(1, color="0.4", ls=":"); axes[1].axvline(np.exp(W_TRUE[1]), color="k", ls="--", lw=1, label="true OR")
axes[1].set_xlabel("odds ratio per allele, exp(w1)"); axes[1].set_ylabel("density"); axes[1].legend(fontsize=7)
axes[1].set_title("posterior of the odds ratio")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The MH samples cover the grid posterior, and transforming each sample gives the posterior of
the odds ratio directly — no delta method or extra derivation needed. The posterior says the allele increases the odds of
disease with probability about 0.99, and gives a credible interval for *how much* that contains the true value of this
simulation. Note that the joint posterior of $(w_0,w_1)$ is tilted: baseline risk and allele effect are correlated in the
posterior — a joint proposal (MH) handles that as easily as an axis-parallel one.

✏️ **Your turn.** Reduce the study to 40 individuals (first line of 8.1) and re-run 8.1–8.3. How wide is the credible
interval of the odds ratio now, and how does $P(\text{OR}>1\mid D)$ change? Which proposal width works best now?

### 📑 Slide connection — `013_Bayesian-MCMC_mz_032725`, slides 44–45

- **What the slides show.** In complex models direct sampling from the posterior is not possible; sampling from a conditional posterior
(one variable given all others) may be — Gibbs sampling; for even more complex models other MCMC methods approximate the
conditional posterior with a tractable distribution (Gaussian, uniform) to draw samples (slide 44); Gibbs sampling versus
Metropolis–Hastings, which samples multiple dimensions at once (slide 45).
- **What this notebook adds.** The Metropolis–Hastings acceptance rule and why $P(D)$ cancels, a from-scratch random-walk sampler for a
logistic-regression posterior (a model with no conjugate prior), a grid reference to verify it, the effect of the proposal
width on acceptance, mixing and effective sample size, and a derived posterior (odds ratio). Clarification of slide 44:
the "tractable distribution" is the proposal; the accept/reject step makes the chain target the exact posterior rather
than an approximation.
- **Exam tip.** Be able to state the MH acceptance probability, explain why the normalising constant is not needed, and
describe what goes wrong with too-small and too-large proposal steps.

---
## 9 · Gibbs sampling for a mixture model, and MCMC in biology (slide 46)

> 📍 **Where we are.** Module 13 fitted mixture models with EM: alternate between *soft* cluster responsibilities
> (E-step) and parameter updates (M-step). The course's third recurring idea is that latent variables lead to such
> alternating schemes; Gibbs sampling is the **sampling sibling** of EM.

Take a marker gene whose log-expression has been measured in 300 cells from two cell types, with unknown proportions and
unknown mean expression in each type (variances fixed to 1 for simplicity). The unknowns are the cell-type labels
$z_i\in\{0,1\}$, the means $\mu_0,\mu_1$ and the proportion $\pi$ of type 1. Every full conditional is easy:

$$P(z_i=1\mid x_i,\mu,\pi)=\frac{\pi\,\mathcal N(x_i;\mu_1,1)}{(1-\pi)\,\mathcal N(x_i;\mu_0,1)+\pi\,\mathcal N(x_i;\mu_1,1)},\qquad
\mu_k\mid z,x\sim\mathcal N\Big(\frac{\sum_{i:z_i=k}x_i+m_0/s_0^2}{n_k+1/s_0^2},\;\frac{1}{n_k+1/s_0^2}\Big),\qquad
\pi\mid z\sim\text{Beta}(1+n_1,\,1+n_0).$$

*In words:* (1) each cell's label is drawn with exactly EM's responsibility as the probability; (2) each mean is drawn
around the average of the cells currently labelled with that type (shrunk slightly towards the prior mean $m_0$ with prior
variance $s_0^2$; $n_k$ is the number of cells with label $k$); (3) the proportion is drawn from a Beta posterior built from
the label counts — Section 2's conjugate pair again. EM *averages* over the labels; Gibbs *samples* them.

In [ ]:
# ── 9.1 Gibbs sampling versus EM for a two-cell-type mixture (SIMULATED expression) ──
rng = np.random.default_rng(SEED + 9)
x_expr = np.r_[rng.normal(2.0, 1.0, 180), rng.normal(5.0, 1.0, 120)]   # 180 type-0 cells, 120 type-1 cells
M0, S0_2 = 3.5, 10.0                                                   # prior on each mean: N(3.5, 10)


def label_probs(x, mu, pi):
    """P(z_i = 1 | x_i, μ, π) for unit-variance components, computed in log space."""
    logp = np.c_[np.log(1 - pi) + stats.norm.logpdf(x, mu[0], 1.0), np.log(pi) + stats.norm.logpdf(x, mu[1], 1.0)]
    return np.exp(logp[:, 1] - logsumexp(logp, axis=1))


def gibbs_mixture(x, n_iter: int, burn: int, rng):
    """Gibbs sampler for (z, μ0, μ1, π); returns parameter samples and each cell's frequency of label 1."""
    mu, pi = np.array([x.min(), x.max()]), 0.5
    params, label_freq = np.empty((n_iter, 3)), np.zeros(len(x))
    for t in range(n_iter):
        z = (rng.random(len(x)) < label_probs(x, mu, pi)).astype(int)        # (1) sample labels
        for c in (0, 1):                                                     # (2) sample means
            prec = np.sum(z == c) + 1 / S0_2
            mu[c] = (x[z == c].sum() + M0 / S0_2) / prec + rng.standard_normal() / np.sqrt(prec)
        pi = rng.beta(1 + z.sum(), 1 + len(x) - z.sum())                    # (3) sample the proportion
        params[t] = mu[0], mu[1], pi
        if t >= burn:
            label_freq += z
    return params, label_freq / (n_iter - burn)


def em_mixture(x, n_iter: int = 200):
    """EM for the same model (Module 13): soft responsibilities instead of sampled labels."""
    mu, pi = np.array([x.min(), x.max()]), 0.5
    for _ in range(n_iter):
        r = label_probs(x, mu, pi)                                           # E-step
        mu = np.array([np.sum((1 - r) * x) / np.sum(1 - r), np.sum(r * x) / np.sum(r)])   # M-step
        pi = r.mean()
    return mu, pi, label_probs(x, mu, pi)


params, freq1 = gibbs_mixture(x_expr, 2000, 200, rng)
post = params[200:]
mu_em, pi_em, resp_em = em_mixture(x_expr)
for name, c, truth in [("μ0", 0, 2.0), ("μ1", 1, 5.0), ("π", 2, 0.4)]:
    print(f"{name}: Gibbs mean {post[:, c].mean():.3f}, 95 % interval [{np.percentile(post[:, c], 2.5):.3f}, "
          f"{np.percentile(post[:, c], 97.5):.3f}]   EM {[mu_em[0], mu_em[1], pi_em][c]:.3f}   truth {truth}")
print(f"max |Gibbs label frequency − EM responsibility| over 300 cells = {np.abs(freq1 - resp_em).max():.3f}")

In [ ]:
# ── 9.2 Pictures: traces, fitted mixture, and labels versus responsibilities ──
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].plot(params[:, 0], lw=0.6, label="μ0"); axes[0].plot(params[:, 1], lw=0.6, label="μ1")
axes[0].axvline(200, color="0.5", ls=":", lw=1); axes[0].set_xlabel("iteration"); axes[0].set_ylabel("mean expression")
axes[0].set_title("Gibbs traces (dotted: end of burn-in)"); axes[0].legend(fontsize=7)
xs = np.linspace(-1.5, 8.5, 300)
m0_hat, m1_hat, pi_hat = post.mean(axis=0)
axes[1].hist(x_expr, bins=40, density=True, color="0.8")
axes[1].plot(xs, (1 - pi_hat) * stats.norm.pdf(xs, m0_hat, 1), color="tab:blue", label="type 0")
axes[1].plot(xs, pi_hat * stats.norm.pdf(xs, m1_hat, 1), color="tab:orange", label="type 1")
axes[1].set_xlabel("log expression of the marker gene"); axes[1].set_ylabel("density")
axes[1].set_title("SIMULATED cells: posterior-mean mixture"); axes[1].legend(fontsize=7)
axes[2].scatter(resp_em, freq1, s=8, color="tab:purple", alpha=0.6)
axes[2].plot([0, 1], [0, 1], "k--", lw=1)
axes[2].set_xlabel("EM responsibility P(type 1 | x)"); axes[2].set_ylabel("Gibbs: fraction of samples with z = 1")
axes[2].set_title("soft labels (EM) ≈ label frequencies (Gibbs)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The Gibbs chain moves from its crude start (the smallest and largest values) to the right
region within a few iterations and then fluctuates around the true means; its posterior means agree with the EM point
estimates, but Gibbs also delivers 95 % intervals for every parameter. Each cell's frequency of being labelled "type 1"
across the Gibbs samples is almost exactly its EM responsibility — the small differences arise because Gibbs also averages
over the uncertainty in $\mu$ and $\pi$, while EM plugs in single values.

⚠️ **Label switching.** The likelihood does not change if we swap the names "type 0" and "type 1". A long chain on a
poorly separated mixture can swap labels mid-run, making the trace of $\mu_0$ jump to $\mu_1$'s value; posterior averages
of $\mu_0$ then become meaningless. Here the types are well separated and the start is ordered, so no switch occurs;
in practice one imposes an ordering ($\mu_0<\mu_1$) or relabels samples afterwards.

### MCMC in biology (slide 46)

Slide 46 lists three classic uses, all of which follow the pattern above — latent variables plus parameters, sampled in
turn:

- **Hidden Markov models — genome phasing.** Which alleles of a person sit together on the same chromosome copy
  (haplotypes)? PHASE (Stephens, Smith & Donnelly 2001) samples each individual's haplotypes given everyone else's.
- **Mixture models — clustering genes, individuals, tumours.** Exactly Section 9, with many components and dimensions;
  Bayesian mixtures also give the uncertainty of each sample's cluster assignment.
- **Admixture models for population structure.** STRUCTURE (Pritchard, Stephens & Donnelly 2000) uses MCMC to infer, for
  every individual, the fractions of its genome inherited from $K$ ancestral populations — the Bayesian cousin of the PCA
  "genes mirror geography" picture of Module 14.

✏️ **Your turn.** Move the two cell types closer (means 2.0 and 3.0 instead of 2.0 and 5.0) and re-run 9.1–9.2. What
happens to the posterior intervals, to the agreement between Gibbs and EM, and to the traces? Do you see label switching?

### 📑 Slide connection — `013_Bayesian-MCMC_mz_032725`, slides 46–47

- **What the slides show.** Gibbs sampling or MCMC is widely used for Bayesian learning and inference: hidden Markov models (genome
phasing), mixture models (clustering genes, individuals, tumours), probabilistic graphical models for population structure
(admixture model) (slide 46); summary — Bayesian learning, the fully Bayesian approach to learn the posterior
distribution, Gibbs sampling as a special case of MCMC (slide 47).
- **What this notebook adds.** A complete Gibbs sampler for a two-component mixture compared with EM (posterior intervals; label frequencies
versus responsibilities), the label-switching pitfall, and pointers to the published tools behind each of slide 46's
applications.
- **Exam tip.** Be ready to contrast EM and Gibbs for mixtures: EM computes expected labels and returns a point estimate
(a local optimum of the likelihood); Gibbs samples labels and parameters and returns a posterior distribution.

## 📌 Key takeaways

- **Three ways to learn a parameter:** MLE maximises the likelihood, MAP maximises the posterior, and the **fully Bayesian**
  approach keeps the whole posterior $P(\theta\mid D)\propto P(D\mid\theta)P(\theta)$ — and with it, uncertainty.
- **Conjugate priors give closed-form posteriors:** Beta prior + binomial likelihood → Beta posterior; Gaussian prior +
  Gaussian likelihood → Gaussian posterior. Credible intervals and posterior predictive distributions follow directly.
- **Bayesian linear regression:** $w\mid y,X\sim\mathcal N(\mu,\Sigma)$ with $\Sigma=(X^\top X/\sigma_e^2+I/\sigma_w^2)^{-1}$
  and $\mu=\Sigma X^\top y/\sigma_e^2$; the mean is the ridge/MAP estimate ($\lambda=\sigma_e^2/\sigma_w^2$), and the
  covariance reveals ambiguities such as linked SNPs.
- **General recipe:** model → prior → posterior. When the posterior has no closed form, **describe it with samples**: any
  expectation, interval or probability becomes an average over samples.
- **Gibbs sampling** draws each variable from its full conditional given all the others; it is the sampling counterpart of
  coordinate descent, needs only one-dimensional conditionals, and mixes slowly when variables are strongly correlated.
- **Practicalities:** discard a **burn-in**, check **trace plots** and running averages, account for **autocorrelation**
  (effective sample size), and compare **several chains** ($\hat R$).
- **Metropolis–Hastings** proposes moves (possibly in many dimensions at once) and accepts them with probability
  $\min(1,p(w')/p(w))$ — the normalising constant cancels; the proposal width trades acceptance against step size. Gibbs
  is a special case of MCMC (slide 47).
- **Gibbs for mixtures is the sampling sibling of EM**, and MCMC powers phasing, Bayesian clustering and admixture
  analysis in genetics.

---
## ✅ Checkpoint

1. A variant is seen on 2 of 40 chromosomes. With a uniform Beta(1, 1) prior, what are the posterior distribution, its mean and its mode? Compare with the MLE.
   <details><summary>Answer</summary>Posterior Beta(1+2, 1+38) = Beta(3, 39). Mean $3/42=0.071$; mode $(3-1)/(3+39-2)=2/40=0.05$, which equals the MLE $2/40$ because the prior is uniform.</details>
2. In Bayesian linear regression, what happens to $\Sigma$ and $\mu$ as $\sigma_w^2\to\infty$, and as $\sigma_w^2\to0$?
   <details><summary>Answer</summary>$\sigma_w^2\to\infty$ (flat prior): $\Sigma\to\sigma_e^2(X^\top X)^{-1}$ and $\mu\to(X^\top X)^{-1}X^\top y$, the OLS estimate with its usual covariance. $\sigma_w^2\to0$ (infinitely strong prior): $\Sigma\to0$ and $\mu\to0$ — the data are ignored and the weights are pinned at the prior mean.</details>
3. Show that the posterior mean $\mu$ equals the ridge estimate with $\lambda=\sigma_e^2/\sigma_w^2$.
   <details><summary>Answer</summary>$\mu=(X^\top X/\sigma_e^2+I/\sigma_w^2)^{-1}X^\top y/\sigma_e^2$. Multiply the matrix inside the inverse by $\sigma_e^2$ and divide the vector by $\sigma_e^2$ (the factors cancel): $\mu=(X^\top X+\frac{\sigma_e^2}{\sigma_w^2}I)^{-1}X^\top y$.</details>
4. Write the full conditional used by Gibbs for $x_2$ in a standard bivariate Gaussian with correlation $\rho=0.9$, given $x_1=1$. Why does this sampler mix slowly?
   <details><summary>Answer</summary>$x_2\mid x_1=1\sim\mathcal N(0.9,\;1-0.81)=\mathcal N(0.9,0.19)$. Each step can move only about $\sqrt{0.19}\approx0.44$ while the distribution stretches much further along the diagonal; successive samples are strongly correlated (lag-1 autocorrelation $\rho^2=0.81$).</details>
5. Code reading: in `gibbs_blr`, why is `r += X[:, j] * w[j]` executed before computing the conditional mean, and `r -= X[:, j] * w[j]` after drawing the new value?
   <details><summary>Answer</summary>`r` holds the full residual $y-Xw$. Adding back predictor $j$'s contribution turns it into the partial residual $r_j=y-\sum_{k\neq j}x_kw_k$ needed for $w_j$'s conditional; after sampling the new $w_j$, subtracting its contribution restores the full residual for the next coordinate — avoiding a full $X w$ product at every step.</details>
6. A Metropolis chain has an acceptance rate of 98 %. Is that good news? What would you change?
   <details><summary>Answer</summary>Usually not: such a high rate means the proposals are tiny, so the chain moves very slowly and its samples are highly autocorrelated (small effective sample size). Increase the proposal step until the acceptance rate is moderate (roughly 20–50 % for random-walk proposals), and check the effective sample size.</details>
7. Why does Metropolis–Hastings not need the evidence $P(D)$?
   <details><summary>Answer</summary>It only uses the ratio $p(w'\mid D)/p(w\mid D)=\frac{P(D\mid w')P(w')/P(D)}{P(D\mid w)P(w)/P(D)}$, in which $P(D)$ cancels; we only need the unnormalised posterior likelihood × prior.</details>
8. What is the difference between EM and Gibbs sampling for a Gaussian mixture, in one sentence each?
   <details><summary>Answer</summary>EM alternates between computing expected (soft) labels and maximising the parameters, returning a single local-optimum point estimate; Gibbs alternates between *sampling* labels and *sampling* parameters from their full conditionals, returning samples from the posterior (with uncertainty).</details>

## 🧠 Questions & discussion

1. Two research groups analyse the same variant with different priors and report different posterior means. When should this worry you, and how would you report results so that readers can judge the influence of the prior?
2. A 95 % credible interval for a SNP effect excludes zero. Is that the same as a significant p-value in a GWAS? What role does the prior play when you test a million SNPs?
3. Gibbs sampling mixes slowly when parameters are strongly correlated. Which models in this course would you expect to have strongly correlated posteriors, and what could you do about it (blocking, reparameterisation, other samplers)?
4. MCMC diagnostics can show non-convergence but cannot prove convergence. Describe a posterior for which four chains could all agree and still be wrong.
5. The fully Bayesian answer is a distribution, but decisions (treat or not, validate this SNP or that one) are discrete. How would you turn a posterior into a decision?

## 🚀 Bonus — going further

- **Blocked Gibbs.** Sample the linked-SNP pair of 3.3 jointly (it is a 2-D Gaussian) instead of one coordinate at a time, and compare effective sample sizes with the one-at-a-time sampler.
- **Hamiltonian Monte Carlo.** Read about HMC, the sampler behind Stan and PyMC, which uses gradients of the log-posterior (Module 04) to make long, well-directed proposals; why does it cope better with correlated posteriors?
- **Bayesian lasso / spike-and-slab.** Replace the Gaussian prior on regression weights by a Laplace prior (whose MAP is the lasso of Module 07) or a spike-and-slab prior, and sample with Gibbs to obtain posterior inclusion probabilities for each predictor — the basis of Bayesian fine-mapping.
- **Uncertainty, another way.** The [ConformalPrediction](../../ConformalPrediction/) tutorial builds prediction intervals with guaranteed frequentist coverage; compare them with the Bayesian predictive bands of 3.4. The [FundamentalsLearningUncertainty](../../FundamentalsLearningUncertainty/) tutorial revisits MLE and MAP.
- **Admixture by Gibbs.** Extend the mixture sampler of Section 9 to genotype data with $K$ ancestral populations (a toy STRUCTURE), using `simulate_genotypes(…, n_pops=3, fst=0.1)` from the toolkit.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 21 · Bayesian learning and MCMC
Input: prostate data (97 men × 8 predictors, real); simulated allele counts, genotype–phenotype, case–control and expression data
Output: posterior distributions (closed form and samples); credible intervals; P(effect > 0); posterior predictive bands
Method: conjugate updates; Bayesian linear regression; direct sampling; Gibbs sampling; random-walk Metropolis–Hastings
Key equation: Σ = (XᵀX/σe² + I/σw²)⁻¹,  μ = Σ Xᵀy/σe²;  accept with prob min(1, p(w′|D)/p(w|D))
Key parameters: prior variance σw²; prior (a, b) or (a0, b0); burn-in; number of iterations; MH proposal width
What changed when I modified the prior / the sample size / the proposal width: …
One failure mode: slow mixing when parameters are strongly correlated (linked SNPs, ρ = 0.98)
Connection to 02-620 slides: 013_Bayesian-MCMC_mz_032725 slides 7–15 (Bayesian regression), 23–35 (Gibbs), 39–45 (burn-in, MH)
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists*, Spring 2025 lecture slides, deck `013_Bayesian-MCMC_mz_032725` (course design S. Kim; instructors M. Xu and M. Zhang).
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*. MIT Press — Chapter 3 (Beta–binomial model), Chapter 7 (Bayesian linear regression), Chapter 24 (Markov chain Monte Carlo).
- Bishop C. M. (2006) *Pattern Recognition and Machine Learning*. Springer — §3.3 (Bayesian linear regression), Chapter 11 (sampling methods).
- Gelman A., Carlin J. B., Stern H. S., Dunson D. B., Vehtari A., Rubin D. B. (2013) *Bayesian Data Analysis*, 3rd ed. CRC Press.
- Hastie T., Tibshirani R., Friedman J. (2009) *The Elements of Statistical Learning*, 2nd ed. Springer — §3.2 (prostate data), §8.6 (MCMC for sampling from the posterior).
- Stamey T. A. et al. (1989) Prostate specific antigen in the diagnosis and treatment of adenocarcinoma of the prostate. II. Radical prostatectomy treated patients. *Journal of Urology* 141:1076–1083.
- Metropolis N., Rosenbluth A. W., Rosenbluth M. N., Teller A. H., Teller E. (1953) Equation of state calculations by fast computing machines. *Journal of Chemical Physics* 21:1087–1092.
- Hastings W. K. (1970) Monte Carlo sampling methods using Markov chains and their applications. *Biometrika* 57:97–109.
- Geman S., Geman D. (1984) Stochastic relaxation, Gibbs distributions, and the Bayesian restoration of images. *IEEE Transactions on Pattern Analysis and Machine Intelligence* 6:721–741.
- Gelfand A. E., Smith A. F. M. (1990) Sampling-based approaches to calculating marginal densities. *Journal of the American Statistical Association* 85:398–409.
- Gelman A., Rubin D. B. (1992) Inference from iterative simulation using multiple sequences. *Statistical Science* 7:457–472.
- Pritchard J. K., Stephens M., Donnelly P. (2000) Inference of population structure using multilocus genotype data. *Genetics* 155:945–959.
- Stephens M., Smith N. J., Donnelly P. (2001) A new statistical method for haplotype reconstruction from population data. *American Journal of Human Genetics* 68:978–989.
- Huelsenbeck J. P., Ronquist F. (2001) MRBAYES: Bayesian inference of phylogenetic trees. *Bioinformatics* 17:754–755.

## ➡️ Next

Bayesian learning treated parameters as uncertain; the next module turns to models with many parameters
and layers. Logistic regression, stacked and trained by gradient descent, becomes a neural network — continue with
[Module 22 · Neural networks: from logistic regression to deep learning on DNA](22_neural_networks.ipynb).